In [1]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 1 - Environment and Google Drive Preflight
# ============================================================

import os
import sys
import platform
import subprocess
from pathlib import Path

import torch
from google.colab import drive


print("=" * 72)
print("MAIA LITE - SFT A100")
print("ENVIRONMENT AND GOOGLE DRIVE PREFLIGHT")
print("=" * 72)
print()


# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

DRIVE_MOUNT = "/content/drive"

print("Mounting Google Drive...")
drive.mount(DRIVE_MOUNT)

print()


# ------------------------------------------------------------
# 2. Project paths
# ------------------------------------------------------------

COLAB_ROOT = Path(
    "/content/drive/MyDrive/Colab_LLMs"
)

SFT_PROJECT = COLAB_ROOT / "maia-sft-validation"

MAIA_PROJECT = COLAB_ROOT / "Maia_Lite_v3"

MAIA_FINAL_CHECKPOINT = (
    MAIA_PROJECT
    / "checkpoints"
    / "checkpoint-216567"
)

STAGE4_DIR = SFT_PROJECT / "data" / "stage4"

BENCHMARK_DIR = STAGE4_DIR / "benchmark"

OUTPUT_DIR = SFT_PROJECT / "outputs"


print("Paths")
print("-" * 72)
print(f"Colab root:            {COLAB_ROOT}")
print(f"SFT project:           {SFT_PROJECT}")
print(f"Maia project:          {MAIA_PROJECT}")
print(f"Maia final checkpoint: {MAIA_FINAL_CHECKPOINT}")
print(f"Stage 4 data:          {STAGE4_DIR}")
print(f"Output directory:      {OUTPUT_DIR}")
print()


# ------------------------------------------------------------
# 3. Environment information
# ------------------------------------------------------------

print("Environment")
print("-" * 72)

print(f"Python:   {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.version.cuda}")
print()


# ------------------------------------------------------------
# 4. CUDA / GPU verification
# ------------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. "
        "Stop here and select an A100 runtime."
    )

device = torch.device("cuda")

gpu_name = torch.cuda.get_device_name(0)

properties = torch.cuda.get_device_properties(0)

gpu_memory_gib = properties.total_memory / (1024 ** 3)

print("GPU")
print("-" * 72)
print(f"Device:       {gpu_name}")
print(f"Memory:       {gpu_memory_gib:.2f} GiB")
print(f"BF16 support: {torch.cuda.is_bf16_supported()}")
print()


# ------------------------------------------------------------
# 5. Require A100
# ------------------------------------------------------------

if "A100" not in gpu_name.upper():
    raise RuntimeError(
        f"Expected NVIDIA A100, but detected: {gpu_name}"
    )

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "The selected GPU does not report BF16 support."
    )


# ------------------------------------------------------------
# 6. Verify required directories
# ------------------------------------------------------------

print("Directory verification")
print("-" * 72)

required_directories = [
    COLAB_ROOT,
    SFT_PROJECT,
    STAGE4_DIR,
    BENCHMARK_DIR,
    OUTPUT_DIR,
]

directory_failure = False

for path in required_directories:
    if path.is_dir():
        print(f"OK       {path}")
    else:
        print(f"MISSING  {path}")
        directory_failure = True

print()

if directory_failure:
    raise RuntimeError(
        "One or more required directories are missing."
    )


# ------------------------------------------------------------
# 7. Verify Stage 4 critical files
# ------------------------------------------------------------

required_files = [
    STAGE4_DIR / "train.jsonl",
    STAGE4_DIR / "validation.jsonl",
    STAGE4_DIR / "test.jsonl",
    BENCHMARK_DIR / "stage4-v4-benchmark.jsonl",
    SFT_PROJECT / "src" / "train_stage4.py",
    SFT_PROJECT / "src" / "train_stage4_lora.py",
    SFT_PROJECT / "requirements-stage4.txt",
]

print("Stage 4 file verification")
print("-" * 72)

file_failure = False

for path in required_files:
    if path.is_file():
        size_mib = path.stat().st_size / (1024 ** 2)
        print(
            f"OK       {path.relative_to(SFT_PROJECT)} "
            f"({size_mib:.2f} MiB)"
        )
    else:
        print(
            f"MISSING  {path.relative_to(SFT_PROJECT)}"
        )
        file_failure = True

print()

if file_failure:
    raise RuntimeError(
        "One or more required Stage 4 files are missing."
    )


# ------------------------------------------------------------
# 8. NVIDIA runtime information
# ------------------------------------------------------------

print("NVIDIA runtime")
print("-" * 72)

subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader",
    ],
    check=True,
)

print()


# ------------------------------------------------------------
# 9. Final status
# ------------------------------------------------------------

print("=" * 72)
print("PREFLIGHT CELL 1: PASS")
print("=" * 72)
print()
print("No training was started.")
print("No project files were modified.")

MAIA LITE - SFT A100
ENVIRONMENT AND GOOGLE DRIVE PREFLIGHT

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Paths
------------------------------------------------------------------------
Colab root:            /content/drive/MyDrive/Colab_LLMs
SFT project:           /content/drive/MyDrive/Colab_LLMs/maia-sft-validation
Maia project:          /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3
Maia final checkpoint: /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/checkpoints/checkpoint-216567
Stage 4 data:          /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/data/stage4
Output directory:      /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs

Environment
------------------------------------------------------------------------
Python:   3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch:  2.11.0+cu130
CUDA:     13.0

GPU
---------------------------------

In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 2 - Frozen Stage 4 SHA-256 Verification
# ============================================================

import hashlib
from pathlib import Path


print("=" * 72)
print("MAIA LITE - SFT A100")
print("FROZEN STAGE 4 SHA-256 VERIFICATION")
print("=" * 72)
print()


SFT_PROJECT = Path(
    "/content/drive/MyDrive/Colab_LLMs/maia-sft-validation"
)

STAGE4_DIR = SFT_PROJECT / "data" / "stage4"


# ------------------------------------------------------------
# Official frozen Stage 4 hashes
# ------------------------------------------------------------

EXPECTED_HASHES = {
    "train.jsonl":
        "ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9",

    "validation.jsonl":
        "5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a",

    "test.jsonl":
        "03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834",

    "benchmark/stage4-v4-benchmark.jsonl":
        "9ba60ac68ad285fdb0e16eb2af876722d8745e8131203b4a090b51b6a5ad73c3",
}


# ------------------------------------------------------------
# SHA-256 helper
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# Verification
# ------------------------------------------------------------

failed = False

for relative_path, expected_hash in EXPECTED_HASHES.items():

    path = STAGE4_DIR / relative_path

    print("-" * 72)
    print(relative_path)

    if not path.is_file():
        print("STATUS:   MISSING")
        failed = True
        continue

    actual_hash = sha256_file(path)

    print(f"Expected: {expected_hash}")
    print(f"Actual:   {actual_hash}")

    if actual_hash == expected_hash:
        print("STATUS:   PASS")
    else:
        print("STATUS:   FAIL")
        failed = True


print()
print("=" * 72)

if failed:
    print("STAGE 4 INTEGRITY VERIFICATION: FAILED")
    print("=" * 72)

    raise RuntimeError(
        "Frozen Stage 4 integrity verification failed. "
        "Do not start training."
    )

else:
    print("STAGE 4 INTEGRITY VERIFICATION: PASS")
    print("=" * 72)
    print()
    print("Frozen Stage 4 artifacts are byte-identical.")
    print("Training may proceed only after the remaining preflight checks.")

MAIA LITE - SFT A100
FROZEN STAGE 4 SHA-256 VERIFICATION

------------------------------------------------------------------------
train.jsonl
Expected: ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9
Actual:   ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9
STATUS:   PASS
------------------------------------------------------------------------
validation.jsonl
Expected: 5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a
Actual:   5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a
STATUS:   PASS
------------------------------------------------------------------------
test.jsonl
Expected: 03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834
Actual:   03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834
STATUS:   PASS
------------------------------------------------------------------------
benchmark/stage4-v4-benchmark.jsonl
Expected: 9ba60ac68ad285fdb0e16eb2af876722d8745e8131203b4a090b51b6a5ad73c3
Actu

In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 3 - Python Dependency Audit
#
# READ-ONLY:
#   - Does not install packages
#   - Does not modify project files
#   - Does not start training
# ============================================================

import sys
import importlib
import importlib.metadata

print("=" * 72)
print("MAIA LITE - SFT A100")
print("PYTHON DEPENDENCY AUDIT")
print("=" * 72)
print()

print(f"Python: {sys.version}")
print()

packages = [
    "torch",
    "transformers",
    "datasets",
    "peft",
    "accelerate",
    "safetensors",
    "tokenizers",
    "tqdm",
    "numpy",
]

print("Installed packages")
print("-" * 72)

missing = []

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package:16s} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package:16s} MISSING")
        missing.append(package)

print()


# ------------------------------------------------------------
# Import verification
# ------------------------------------------------------------

print("Import verification")
print("-" * 72)

import_failed = []

for package in packages:
    try:
        importlib.import_module(package)
        print(f"{package:16s} PASS")
    except Exception as exc:
        print(
            f"{package:16s} FAIL "
            f"({type(exc).__name__}: {exc})"
        )
        import_failed.append(package)

print()


# ------------------------------------------------------------
# PyTorch / CUDA verification
# ------------------------------------------------------------

import torch

print("PyTorch / CUDA")
print("-" * 72)

print(f"PyTorch version:       {torch.__version__}")
print(f"CUDA runtime:          {torch.version.cuda}")
print(f"CUDA available:        {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU:                   {torch.cuda.get_device_name(0)}")
    print(f"BF16 supported:        {torch.cuda.is_bf16_supported()}")
    print(
        f"GPU memory:            "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB"
    )

print()


# ------------------------------------------------------------
# Requirements file
# ------------------------------------------------------------

requirements_file = (
    SFT_PROJECT / "requirements-stage4.txt"
)

print("Project requirements")
print("-" * 72)

if requirements_file.is_file():
    print(requirements_file)
    print()

    contents = requirements_file.read_text(
        encoding="utf-8"
    )

    print(contents)
else:
    print("requirements-stage4.txt: MISSING")

print()


# ------------------------------------------------------------
# Final audit status
# ------------------------------------------------------------

print("=" * 72)

if missing or import_failed:
    print("DEPENDENCY AUDIT: ATTENTION REQUIRED")

    if missing:
        print(
            "Missing packages:",
            ", ".join(missing)
        )

    if import_failed:
        print(
            "Import failures:",
            ", ".join(import_failed)
        )
else:
    print("DEPENDENCY AUDIT: PASS")

print("=" * 72)
print()
print("No packages were installed.")
print("No project files were modified.")
print("No training was started.")

MAIA LITE - SFT A100
PYTHON DEPENDENCY AUDIT

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

Installed packages
------------------------------------------------------------------------
torch            2.11.0+cu130
transformers     5.17.0
datasets         4.8.5
peft             0.21.0
accelerate       1.15.0
safetensors      0.8.0
tokenizers       0.23.2
tqdm             4.67.3
numpy            2.1.3

Import verification
------------------------------------------------------------------------
torch            PASS
transformers     PASS
datasets         PASS
peft             PASS
accelerate       PASS
safetensors      PASS
tokenizers       PASS
tqdm             PASS
numpy            PASS

PyTorch / CUDA
------------------------------------------------------------------------
PyTorch version:       2.11.0+cu130
CUDA runtime:          13.0
CUDA available:        True
GPU:                   NVIDIA A100-SXM4-80GB
BF16 supported:        True
GPU memory:            79.25 GiB

Pro

In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 4 - Stage 4 Training Code Audit
#
# READ-ONLY:
#   - Does not modify scripts
#   - Does not start training
# ============================================================

from pathlib import Path
import hashlib
import re


print("=" * 72)
print("MAIA LITE - SFT A100")
print("STAGE 4 TRAINING CODE AUDIT")
print("=" * 72)
print()


scripts = [
    SFT_PROJECT / "src" / "train_stage4.py",
    SFT_PROJECT / "src" / "train_stage4_lora.py",
    SFT_PROJECT / "src" / "train_stage4_lora_mps_safe.py",
]


def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


patterns = {
    "MPS references": r"\bmps\b|torch\.mps|empty_cache",
    "CUDA references": r"\bcuda\b|torch\.cuda",
    "BF16 references": r"\bbf16\b|bfloat16",
    "FP16 references": r"\bfp16\b|float16",
    "Gradient checkpointing": r"gradient_checkpoint",
    "use_cache": r"use_cache",
    "LoRA": r"\bLoraConfig\b|\bLoRA\b|\blora\b",
    "AdamW": r"\bAdamW\b",
    "batch size": r"batch_size",
    "gradient accumulation": r"gradient_accum",
}


for path in scripts:

    print("-" * 72)
    print(path.name)
    print("-" * 72)

    if not path.is_file():
        print("MISSING")
        print()
        continue

    text = path.read_text(
        encoding="utf-8",
        errors="replace",
    )

    lines = text.splitlines()

    print(f"Size:    {path.stat().st_size:,} bytes")
    print(f"Lines:   {len(lines):,}")
    print(f"SHA-256: {sha256_file(path)}")
    print()

    print("Feature scan:")

    for label, pattern in patterns.items():
        matches = [
            index + 1
            for index, line in enumerate(lines)
            if re.search(
                pattern,
                line,
                flags=re.IGNORECASE,
            )
        ]

        if matches:
            preview = ", ".join(
                str(x) for x in matches[:12]
            )

            if len(matches) > 12:
                preview += ", ..."

            print(
                f"  {label:24s} "
                f"YES  lines: {preview}"
            )
        else:
            print(
                f"  {label:24s} NO"
            )

    print()


print("=" * 72)
print("CODE AUDIT COMPLETE")
print("=" * 72)
print()
print("No training script was executed.")
print("No project file was modified.")

MAIA LITE - SFT A100
STAGE 4 TRAINING CODE AUDIT

------------------------------------------------------------------------
train_stage4.py
------------------------------------------------------------------------
Size:    32,669 bytes
Lines:   1,533
SHA-256: 4a22a08fc9d66fc46924016fa126f728fc79f0a0057c2c6660665e931a405f6a

Feature scan:
  MPS references           YES  lines: 46, 190, 192, 194, 199, 200, 231, 232, 234, 239, 240
  CUDA references          YES  lines: 185, 186, 187, 223, 224, 226, 227
  BF16 references          NO
  FP16 references          NO
  Gradient checkpointing   NO
  use_cache                NO
  LoRA                     NO
  AdamW                    YES  lines: 934
  batch size               YES  lines: 272, 285
  gradient accumulation    YES  lines: 508, 510, 750

------------------------------------------------------------------------
train_stage4_lora.py
------------------------------------------------------------------------
Size:    37,343 bytes
Lines:   1,68

In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 5 - Frozen Training Configuration Inspection
#
# READ-ONLY:
#   - Does not import training scripts
#   - Does not modify files
#   - Does not start training
# ============================================================

from pathlib import Path
import re


print("=" * 72)
print("MAIA LITE - SFT A100")
print("FROZEN TRAINING CONFIGURATION INSPECTION")
print("=" * 72)
print()


scripts = [
    SFT_PROJECT / "src" / "common_stage4.py",
    SFT_PROJECT / "src" / "train_stage4.py",
    SFT_PROJECT / "src" / "train_stage4_lora.py",
]


interesting_patterns = [
    r"CFG\s*=",
    r"model_id",
    r"max_length",
    r"batch_size",
    r"gradient_accumulation",
    r"learning_rate",
    r"weight_decay",
    r"epochs",
    r"warmup",
    r"max_grad_norm",
    r"SEED",
    r"def device",
    r"DataLoader",
    r"num_workers",
    r"pin_memory",
    r"gradient_checkpoint",
    r"use_cache",
    r"LORA_",
]


for path in scripts:

    print()
    print("=" * 72)
    print(path.name)
    print("=" * 72)

    if not path.is_file():
        print("MISSING")
        continue

    lines = path.read_text(
        encoding="utf-8",
        errors="replace",
    ).splitlines()

    matched_lines = set()

    for index, line in enumerate(lines):

        if any(
            re.search(pattern, line)
            for pattern in interesting_patterns
        ):
            # Include surrounding lines so multiline
            # dictionaries/configurations remain readable.
            start = max(0, index - 3)
            end = min(len(lines), index + 8)

            matched_lines.update(
                range(start, end)
            )

    if not matched_lines:
        print("No relevant configuration lines found.")
        continue

    previous = None

    for index in sorted(matched_lines):

        if (
            previous is not None
            and index > previous + 1
        ):
            print("    ...")

        print(
            f"{index + 1:5d}: "
            f"{lines[index]}"
        )

        previous = index


print()
print("=" * 72)
print("CONFIGURATION INSPECTION COMPLETE")
print("=" * 72)
print()
print("No training script was imported.")
print("No project file was modified.")
print("No training was started.")

MAIA LITE - SFT A100
FROZEN TRAINING CONFIGURATION INSPECTION


common_stage4.py
    1: import hashlib,json,random,re
    2: from pathlib import Path
    3: import torch
    4: ROOT=Path(__file__).resolve().parents[1]; CFG=json.loads((ROOT/"configs/stage4.json").read_text()); DATA=ROOT/"data/stage4"; OUT=ROOT/"outputs"; SEED=CFG["seed"]
    5: SYSTEM="You are a helpful, careful multilingual assistant. Answer in the user's language unless asked otherwise."
    6: def seed_all(): random.seed(SEED); torch.manual_seed(SEED)
    7: def device():
    8:     if torch.cuda.is_available(): return torch.device("cuda")
    9:     if torch.backends.mps.is_available(): return torch.device("mps")
   10:     return torch.device("cpu")
   11: def sha256(p):
   12:     h=hashlib.sha256()
   13:     with open(p,"rb") as f:
   14:         for b in iter(lambda:f.read(1048576),b""): h.update(b)

train_stage4.py
    8: from pathlib import Path
    9: 
   10: import torch
   11: from torch.utils.data import 

In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 6 - Exact Frozen Stage 4 Configuration
# ============================================================

import json

config_path = (
    SFT_PROJECT
    / "configs"
    / "stage4.json"
)

print("=" * 72)
print("FROZEN STAGE 4 CONFIGURATION")
print("=" * 72)
print()

with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

print()
print("=" * 72)
print("CONFIGURATION READ COMPLETE")
print("=" * 72)
print()
print("No project file was modified.")
print("No training was started.")

FROZEN STAGE 4 CONFIGURATION

{
  "stage": "stage4-full-assistant-control-v1",
  "model_id": "openai-community/gpt2-medium",
  "seed": 42,
  "max_length": 1024,
  "micro_batch_size": 1,
  "gradient_accumulation_steps": 8,
  "epochs": 3,
  "learning_rate": 2e-05,
  "weight_decay": 0.01,
  "warmup_ratio": 0.05,
  "max_grad_norm": 1.0,
  "objective": "response-only causal language modeling",
  "selection": "best validation loss",
  "languages": [
    "en",
    "pt",
    "es"
  ],
  "target_examples": 18000,
  "benchmark_examples": 300
}

CONFIGURATION READ COMPLETE

No project file was modified.
No training was started.


In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 7 - Create A100 LoRA Training Variant
#
# Creates:
#   src/train_stage4_lora_a100.py
#
# The original train_stage4_lora.py is NEVER modified.
# ============================================================

from pathlib import Path
import hashlib


print("=" * 72)
print("MAIA LITE - SFT A100")
print("CREATE GPT-2 MEDIUM LoRA A100 VARIANT")
print("=" * 72)
print()


SOURCE = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora.py"
)

TARGET = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora_a100.py"
)


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# Verify source
# ------------------------------------------------------------

EXPECTED_SOURCE_SHA256 = (
    "f38ce575754c4da54bbdad4e39886960"
    "c05bc207e2a5c5aae8a74f3bf849a6c4"
)

if not SOURCE.is_file():
    raise RuntimeError(
        f"Source script not found: {SOURCE}"
    )

source_hash = sha256_file(SOURCE)

print("Source:")
print(f"  {SOURCE}")
print()
print("Expected SHA-256:")
print(f"  {EXPECTED_SOURCE_SHA256}")
print()
print("Actual SHA-256:")
print(f"  {source_hash}")
print()

if source_hash != EXPECTED_SOURCE_SHA256:
    raise RuntimeError(
        "Source train_stage4_lora.py does not match "
        "the audited script. A100 variant was NOT created."
    )

print("Source integrity: PASS")
print()


# ------------------------------------------------------------
# Read exact audited source
# ------------------------------------------------------------

text = SOURCE.read_text(
    encoding="utf-8"
)


# ------------------------------------------------------------
# Controlled transformation 1:
# Dedicated A100 output directories.
#
# This prevents collision with M4 experiment artifacts.
# ------------------------------------------------------------

replacements = {
    'CHECKPOINT_ROOT = OUT / "stage4-lora-checkpoints"':
        'CHECKPOINT_ROOT = OUT / "a100-gpt2-medium-lora-checkpoints"',

    'BEST_DIR = OUT / "gpt2-medium-stage4-lora-best"':
        'BEST_DIR = OUT / "a100-gpt2-medium-lora-best"',

    'FINAL_DIR = OUT / "gpt2-medium-stage4-lora-final"':
        'FINAL_DIR = OUT / "a100-gpt2-medium-lora-final"',

    'METRICS_FILE = OUT / "stage4-lora-training-metrics.json"':
        'METRICS_FILE = OUT / "a100-gpt2-medium-lora-training-metrics.json"',
}


for old, new in replacements.items():

    count = text.count(old)

    if count != 1:
        raise RuntimeError(
            f"Expected exactly one occurrence of:\n{old}\n"
            f"Found: {count}\n"
            "A100 variant was NOT created."
        )

    text = text.replace(
        old,
        new,
        1,
    )


# ------------------------------------------------------------
# Controlled transformation 2:
# Require CUDA explicitly.
#
# We deliberately do NOT allow silent fallback to MPS/CPU.
# ------------------------------------------------------------

needle = """    dev = device()

    print()
"""

replacement = """    dev = device()

    if dev.type != "cuda":
        raise RuntimeError(
            f"A100 experiment requires CUDA, got: {dev}"
        )

    gpu_name = torch.cuda.get_device_name(0)

    if "A100" not in gpu_name.upper():
        raise RuntimeError(
            f"A100 experiment requires NVIDIA A100, got: {gpu_name}"
        )

    if not torch.cuda.is_bf16_supported():
        raise RuntimeError(
            "A100 experiment requires BF16 support."
        )

    print()
    print("CUDA device:", gpu_name)
    print(
        "CUDA memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB",
    )
    print("BF16 support:", torch.cuda.is_bf16_supported())

    print()
"""

count = text.count(needle)

if count != 1:
    raise RuntimeError(
        "Could not locate the unique device initialization "
        "block in the audited source."
    )

text = text.replace(
    needle,
    replacement,
    1,
)


# ------------------------------------------------------------
# Controlled transformation 3:
# Experiment banner only.
# ------------------------------------------------------------

text = text.replace(
    "GPT-2 MEDIUM RESPONSE-ONLY LoRA SFT",
    "GPT-2 MEDIUM RESPONSE-ONLY LoRA SFT - A100 CONTROL",
    1,
)


# ------------------------------------------------------------
# Write atomically
# ------------------------------------------------------------

temporary = TARGET.with_suffix(
    TARGET.suffix + ".tmp"
)

temporary.write_text(
    text,
    encoding="utf-8",
)

temporary.replace(TARGET)


# ------------------------------------------------------------
# Verify result
# ------------------------------------------------------------

target_hash = sha256_file(TARGET)

print("A100 variant created:")
print(f"  {TARGET}")
print()

print("SHA-256:")
print(f"  {target_hash}")
print()

print("Output isolation:")
print("  checkpoints : outputs/a100-gpt2-medium-lora-checkpoints")
print("  best        : outputs/a100-gpt2-medium-lora-best")
print("  final       : outputs/a100-gpt2-medium-lora-final")
print(
    "  metrics     : "
    "outputs/a100-gpt2-medium-lora-training-metrics.json"
)

print()
print("Frozen experimental parameters:")
print("  model                    : openai-community/gpt2-medium")
print("  seed                     : 42")
print("  max_length               : 1024")
print("  micro_batch_size         : 1")
print("  gradient accumulation    : 8")
print("  effective batch          : 8")
print("  epochs                   : 3")
print("  learning rate            : 2e-5")
print("  weight decay             : 0.01")
print("  warmup ratio             : 0.05")
print("  max grad norm            : 1.0")
print("  LoRA rank                : 16")
print("  LoRA alpha               : 32")
print("  LoRA dropout             : 0.05")
print("  LoRA targets             : c_attn, c_proj, c_fc")
print("  gradient checkpointing   : preserved")
print("  use_cache=False          : preserved")

print()
print("=" * 72)
print("A100 VARIANT CREATION: PASS")
print("=" * 72)
print()
print("The original M4 training scripts were not modified.")
print("Training was NOT started.")

MAIA LITE - SFT A100
CREATE GPT-2 MEDIUM LoRA A100 VARIANT

Source:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/src/train_stage4_lora.py

Expected SHA-256:
  f38ce575754c4da54bbdad4e39886960c05bc207e2a5c5aae8a74f3bf849a6c4

Actual SHA-256:
  f38ce575754c4da54bbdad4e39886960c05bc207e2a5c5aae8a74f3bf849a6c4

Source integrity: PASS

A100 variant created:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/src/train_stage4_lora_a100.py

SHA-256:
  80edb17ae5c03c93cddc03db8bea3ad962908d9921d65d7b95c5f4e0398d8a4d

Output isolation:
  checkpoints : outputs/a100-gpt2-medium-lora-checkpoints
  best        : outputs/a100-gpt2-medium-lora-best
  final       : outputs/a100-gpt2-medium-lora-final
  metrics     : outputs/a100-gpt2-medium-lora-training-metrics.json

Frozen experimental parameters:
  model                    : openai-community/gpt2-medium
  seed                     : 42
  max_length               : 1024
  micro_batch_size         : 1
  gradient accumulation    : 8
  ef

In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 8 - Validate A100 LoRA Variant
#
# READ-ONLY:
#   - Python syntax check
#   - Unified diff against audited source
#   - Safety checks
#   - Does NOT import or execute training code
# ============================================================

from pathlib import Path
import difflib
import hashlib
import py_compile


print("=" * 72)
print("MAIA LITE - SFT A100")
print("VALIDATE GPT-2 MEDIUM LoRA A100 VARIANT")
print("=" * 72)
print()


SOURCE = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora.py"
)

TARGET = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora_a100.py"
)


EXPECTED_SOURCE_SHA256 = (
    "f38ce575754c4da54bbdad4e39886960"
    "c05bc207e2a5c5aae8a74f3bf849a6c4"
)

EXPECTED_TARGET_SHA256 = (
    "80edb17ae5c03c93cddc03db8bea3ad9"
    "62908d9921d65d7b95c5f4e0398d8a4d"
)


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# 1. Integrity
# ------------------------------------------------------------

source_hash = sha256_file(SOURCE)
target_hash = sha256_file(TARGET)

print("Integrity")
print("-" * 72)

print(f"Source: {source_hash}")
print(f"Target: {target_hash}")

if source_hash != EXPECTED_SOURCE_SHA256:
    raise RuntimeError(
        "Original audited source changed."
    )

if target_hash != EXPECTED_TARGET_SHA256:
    raise RuntimeError(
        "Generated A100 variant changed."
    )

print("STATUS: PASS")
print()


# ------------------------------------------------------------
# 2. Python syntax compilation
# ------------------------------------------------------------

print("Python syntax")
print("-" * 72)

try:
    py_compile.compile(
        str(TARGET),
        doraise=True,
    )
except py_compile.PyCompileError as exc:
    raise RuntimeError(
        f"A100 variant does not compile:\n{exc}"
    )

print("STATUS: PASS")
print()


# ------------------------------------------------------------
# 3. Read files
# ------------------------------------------------------------

source_lines = SOURCE.read_text(
    encoding="utf-8"
).splitlines(keepends=True)

target_lines = TARGET.read_text(
    encoding="utf-8"
).splitlines(keepends=True)

target_text = "".join(target_lines)


# ------------------------------------------------------------
# 4. Safety assertions
# ------------------------------------------------------------

print("Safety assertions")
print("-" * 72)

required_strings = [
    'a100-gpt2-medium-lora-checkpoints',
    'a100-gpt2-medium-lora-best',
    'a100-gpt2-medium-lora-final',
    'a100-gpt2-medium-lora-training-metrics.json',
    'A100 experiment requires CUDA',
    'A100 experiment requires NVIDIA A100',
    'A100 experiment requires BF16 support',
]

for value in required_strings:
    if value not in target_text:
        raise RuntimeError(
            f"Required A100 marker missing: {value}"
        )

    print(f"PASS  {value}")


for old_value in [
    'OUT / "stage4-lora-checkpoints"',
    'OUT / "gpt2-medium-stage4-lora-best"',
    'OUT / "gpt2-medium-stage4-lora-final"',
    'OUT / "stage4-lora-training-metrics.json"',
]:
    if old_value in target_text:
        raise RuntimeError(
            f"Old M4 output path remains active: {old_value}"
        )

print()
print("M4 output isolation: PASS")
print()


# ------------------------------------------------------------
# 5. Show complete unified diff
# ------------------------------------------------------------

print("=" * 72)
print("UNIFIED DIFF")
print("=" * 72)
print()

diff = list(
    difflib.unified_diff(
        source_lines,
        target_lines,
        fromfile="train_stage4_lora.py",
        tofile="train_stage4_lora_a100.py",
    )
)

if not diff:
    raise RuntimeError(
        "No differences found between source and A100 variant."
    )

print("".join(diff))


# ------------------------------------------------------------
# 6. Diff statistics
# ------------------------------------------------------------

added = sum(
    1
    for line in diff
    if line.startswith("+")
    and not line.startswith("+++")
)

removed = sum(
    1
    for line in diff
    if line.startswith("-")
    and not line.startswith("---")
)

print()
print("=" * 72)
print("DIFF STATISTICS")
print("=" * 72)
print(f"Added lines:   {added}")
print(f"Removed lines: {removed}")
print()


# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print("=" * 72)
print("A100 VARIANT STRUCTURAL VALIDATION: PASS")
print("=" * 72)
print()
print("Original M4 script integrity: PASS")
print("A100 script integrity:        PASS")
print("Python syntax:                PASS")
print("Output isolation:             PASS")
print()
print("Training was NOT started.")

MAIA LITE - SFT A100
VALIDATE GPT-2 MEDIUM LoRA A100 VARIANT

Integrity
------------------------------------------------------------------------
Source: f38ce575754c4da54bbdad4e39886960c05bc207e2a5c5aae8a74f3bf849a6c4
Target: 80edb17ae5c03c93cddc03db8bea3ad962908d9921d65d7b95c5f4e0398d8a4d
STATUS: PASS

Python syntax
------------------------------------------------------------------------
STATUS: PASS

Safety assertions
------------------------------------------------------------------------
PASS  a100-gpt2-medium-lora-checkpoints
PASS  a100-gpt2-medium-lora-best
PASS  a100-gpt2-medium-lora-final
PASS  a100-gpt2-medium-lora-training-metrics.json
PASS  A100 experiment requires CUDA
PASS  A100 experiment requires NVIDIA A100
PASS  A100 experiment requires BF16 support

M4 output isolation: PASS

UNIFIED DIFF

--- train_stage4_lora.py
+++ train_stage4_lora_a100.py
@@ -32,7 +32,7 @@
 # MAIA SFT VALIDATION
 # STAGE 4 - FULL ASSISTANT CONTROL
 #
-# GPT-2 MEDIUM RESPONSE-ONLY LoRA SFT
+# GPT-

In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 9 - GPT-2 Medium Stage 4 LoRA
#
# CONTROL EXPERIMENT
#   Device: NVIDIA A100
#   Precision: FP32
#   Context: 1024
#   Micro batch: 1
#   Gradient accumulation: 8
#   Effective batch: 8
#   Epochs: 3
#
# The training script performs:
#   - frozen corpus verification
#   - A100/CUDA verification
#   - initial validation
#   - LoRA training
#   - periodic resumable checkpoints
#   - validation after every epoch
#   - best/final adapter preservation
# ============================================================

import os
import subprocess
import sys
from pathlib import Path


print("=" * 72)
print("MAIA LITE - SFT A100")
print("GPT-2 MEDIUM STAGE 4 LoRA CONTROL")
print("=" * 72)
print()

script = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora_a100.py"
)

if not script.is_file():
    raise RuntimeError(
        f"Training script not found: {script}"
    )

print("Project:")
print(f"  {SFT_PROJECT}")

print()
print("Training script:")
print(f"  {script}")

print()
print("Working directory:")
print(f"  {SFT_PROJECT}")

print()
print("Experiment:")
print("  Base model:             openai-community/gpt2-medium")
print("  Device:                 NVIDIA A100")
print("  Precision:              FP32")
print("  Context:                1024")
print("  Micro batch:            1")
print("  Gradient accumulation:  8")
print("  Effective batch:        8")
print("  Epochs:                 3")
print("  LoRA r/alpha/dropout:   16 / 32 / 0.05")

print()
print("=" * 72)
print("STARTING TRAINING")
print("=" * 72)
print()

env = os.environ.copy()

env["PYTHONUNBUFFERED"] = "1"

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "-m",
        "src.train_stage4_lora_a100",
    ],
    cwd=str(SFT_PROJECT),
    env=env,
)

print()
print("=" * 72)

if result.returncode == 0:
    print("GPT-2 MEDIUM LoRA A100: COMPLETED")
else:
    print(
        "GPT-2 MEDIUM LoRA A100: FAILED "
        f"(exit code {result.returncode})"
    )

print("=" * 72)

if result.returncode != 0:
    raise RuntimeError(
        "A100 LoRA training process failed."
    )

MAIA LITE - SFT A100
GPT-2 MEDIUM STAGE 4 LoRA CONTROL

Project:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation

Training script:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/src/train_stage4_lora_a100.py

Working directory:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation

Experiment:
  Base model:             openai-community/gpt2-medium
  Device:                 NVIDIA A100
  Precision:              FP32
  Context:                1024
  Micro batch:            1
  Gradient accumulation:  8
  Effective batch:        8
  Epochs:                 3
  LoRA r/alpha/dropout:   16 / 32 / 0.05

STARTING TRAINING



KeyboardInterrupt: 

In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Diagnostic - Import A100 Training Module
#
# Does NOT call main()
# Does NOT start training
# ============================================================

import os
import subprocess
import sys

print("=" * 72)
print("A100 LoRA - IMPORT DIAGNOSTIC")
print("=" * 72)
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

code = """
import sys
import traceback

print("Python:", sys.version)
print()

try:
    import src.train_stage4_lora_a100
    print("MODULE IMPORT: PASS")
except Exception:
    print("MODULE IMPORT: FAILED")
    print()
    traceback.print_exc()
    sys.exit(1)
"""

result = subprocess.run(
    [
        sys.executable,
        "-u",
        "-c",
        code,
    ],
    cwd=str(SFT_PROJECT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)

print("=" * 72)
print("Exit code:", result.returncode)
print("=" * 72)

A100 LoRA - IMPORT DIAGNOSTIC

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

MODULE IMPORT: PASS

Exit code: 0


In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 9B - Run With Full Error Visibility
#
# Runs the SAME validated A100 training script.
# No source file is modified.
# ============================================================

import os
import subprocess
import sys


print("=" * 72)
print("MAIA LITE - SFT A100")
print("GPT-2 MEDIUM LoRA - FULL DIAGNOSTIC OUTPUT")
print("=" * 72)
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "-m",
        "src.train_stage4_lora_a100",
    ],
    cwd=str(SFT_PROJECT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in process.stdout:
        print(line, end="", flush=True)

finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()

print()
print("=" * 72)
print(f"PROCESS EXIT CODE: {return_code}")
print("=" * 72)

if return_code != 0:
    print()
    print(
        "Training process failed. "
        "The original traceback should now be visible above."
    )
else:
    print()
    print("Training completed successfully.")

MAIA LITE - SFT A100
GPT-2 MEDIUM LoRA - FULL DIAGNOSTIC OUTPUT


VERIFYING FROZEN STAGE 4 V4 CORPUS

train.jsonl
  expected: ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9
  actual:   ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9

validation.jsonl
  expected: 5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a
  actual:   5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a

test.jsonl
  expected: 03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834
  actual:   03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834

Frozen corpus verification: PASS

CUDA device: NVIDIA A100-SXM4-80GB
CUDA memory: 79.25 GiB
BF16 support: True

STAGE 4 LoRA TRAINING

Device: cuda
Model:  openai-community/gpt2-medium

Loading datasets...
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1317 > 1024). Running this sequence through the model will result in inde

In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 10 - Remove Incompatible Optional torchao Package
#
# PEFT 0.21.0 detects torchao 0.10.0, but requires > 0.16.0.
# Stage 4 LoRA does not use torchao.
# ============================================================

import subprocess
import sys

print("=" * 72)
print("REMOVE INCOMPATIBLE OPTIONAL TORCHAO")
print("=" * 72)
print()

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=True,
)

print()
print("=" * 72)
print("TORCHAO REMOVAL: COMPLETE")
print("=" * 72)
print()
print("Restart the Colab runtime before continuing.")

REMOVE INCOMPATIBLE OPTIONAL TORCHAO


TORCHAO REMOVAL: COMPLETE

Restart the Colab runtime before continuing.


In [ ]:
# ============================================================
# Check torchao state in a fresh Python subprocess
# ============================================================

import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-c",
        """
import importlib.metadata

try:
    version = importlib.metadata.version("torchao")
    print("torchao:", version)
except importlib.metadata.PackageNotFoundError:
    print("torchao: NOT INSTALLED")

import torch
import transformers
import peft

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
"""
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print(result.stdout)
print("Exit code:", result.returncode)

torchao: NOT INSTALLED
torch: 2.11.0+cu130
transformers: 5.17.0
peft: 0.21.0
CUDA: 13.0
GPU: NVIDIA A100-SXM4-80GB

Exit code: 0


In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Post-Restart Recovery and Integrity Check
# ============================================================

from google.colab import drive
from pathlib import Path
import hashlib
import importlib.metadata
import torch


print("=" * 72)
print("MAIA LITE - SFT A100")
print("POST-RESTART RECOVERY")
print("=" * 72)
print()


# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

COLAB_ROOT = Path(
    "/content/drive/MyDrive/Colab_LLMs"
)

SFT_PROJECT = (
    COLAB_ROOT
    / "maia-sft-validation"
)

STAGE4 = (
    SFT_PROJECT
    / "data"
    / "stage4"
)

A100_SCRIPT = (
    SFT_PROJECT
    / "src"
    / "train_stage4_lora_a100.py"
)


# ------------------------------------------------------------
# SHA-256 helper
# ------------------------------------------------------------

def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------

print()
print("Environment")
print("-" * 72)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available."
    )

gpu_name = torch.cuda.get_device_name(0)

print("PyTorch:     ", torch.__version__)
print("CUDA:        ", torch.version.cuda)
print("GPU:         ", gpu_name)
print(
    "GPU memory:  ",
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB",
)
print(
    "BF16:        ",
    torch.cuda.is_bf16_supported(),
)

if "A100" not in gpu_name.upper():
    raise RuntimeError(
        f"Expected A100, got: {gpu_name}"
    )


# ------------------------------------------------------------
# Python package state
# ------------------------------------------------------------

print()
print("Packages")
print("-" * 72)

for package in [
    "transformers",
    "peft",
    "accelerate",
]:
    version = importlib.metadata.version(package)

    print(
        f"{package:14s} {version}"
    )


try:
    torchao_version = (
        importlib.metadata.version("torchao")
    )

    raise RuntimeError(
        "torchao is still installed: "
        f"{torchao_version}"
    )

except importlib.metadata.PackageNotFoundError:
    print(
        f"{'torchao':14s} NOT INSTALLED — PASS"
    )


# ------------------------------------------------------------
# Frozen Stage 4 hashes
# ------------------------------------------------------------

print()
print("Frozen Stage 4")
print("-" * 72)

expected_hashes = {
    STAGE4 / "train.jsonl":
        "ee80a1efc74807d354761d9728cd498e9"
        "f543c84a5e568e4f3821d2fa48565f9",

    STAGE4 / "validation.jsonl":
        "5d8ea877e14105e06068f2d7de5ae55d"
        "265b4069af2c1b0762b162df1be1347a",

    STAGE4 / "test.jsonl":
        "03b7289a5841f79e1c95188f22615dca"
        "6789846aeddecc1b3c90f9713c918834",

    STAGE4 / "benchmark" / "stage4-v4-benchmark.jsonl":
        "9ba60ac68ad285fdb0e16eb2af876722d"
        "8745e8131203b4a090b51b6a5ad73c3",
}

for path, expected in expected_hashes.items():

    if not path.is_file():
        raise RuntimeError(
            f"Missing frozen artifact: {path}"
        )

    actual = sha256_file(path)

    status = (
        "PASS"
        if actual == expected
        else "FAIL"
    )

    print(
        f"{path.name:32s} {status}"
    )

    if actual != expected:
        raise RuntimeError(
            f"Hash mismatch: {path}"
        )


# ------------------------------------------------------------
# A100 training script
# ------------------------------------------------------------

print()
print("A100 training script")
print("-" * 72)

EXPECTED_A100_SHA256 = (
    "80edb17ae5c03c93cddc03db8bea3ad9"
    "62908d9921d65d7b95c5f4e0398d8a4d"
)

if not A100_SCRIPT.is_file():
    raise RuntimeError(
        f"Missing A100 script: {A100_SCRIPT}"
    )

actual_script_hash = sha256_file(
    A100_SCRIPT
)

print("Expected:")
print(EXPECTED_A100_SHA256)

print()
print("Actual:")
print(actual_script_hash)

if actual_script_hash != EXPECTED_A100_SHA256:
    raise RuntimeError(
        "A100 script hash mismatch."
    )

print()
print("A100 script integrity: PASS")


# ------------------------------------------------------------
# Final result
# ------------------------------------------------------------

print()
print("=" * 72)
print("POST-RESTART RECOVERY: PASS")
print("=" * 72)
print()
print("Environment recovered.")
print("Frozen Stage 4 corpus verified.")
print("A100 training script verified.")
print("Incompatible torchao absent.")
print()
print("Training was NOT started.")

MAIA LITE - SFT A100
POST-RESTART RECOVERY

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Environment
------------------------------------------------------------------------
PyTorch:      2.11.0+cu130
CUDA:         13.0
GPU:          NVIDIA A100-SXM4-80GB
GPU memory:   79.25 GiB
BF16:         True

Packages
------------------------------------------------------------------------
transformers   5.17.0
peft           0.21.0
accelerate     1.15.0
torchao        NOT INSTALLED — PASS

Frozen Stage 4
------------------------------------------------------------------------
train.jsonl                      PASS
validation.jsonl                 PASS
test.jsonl                       PASS
stage4-v4-benchmark.jsonl        PASS

A100 training script
------------------------------------------------------------------------
Expected:
80edb17ae5c03c93cddc03db8bea3ad962908d9921d65d7b95c5f4e0398d8a4d

Actual:
80edb17ae5c

In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Cell 9B - GPT-2 Medium Stage 4 LoRA
# Real-Time Output
#
# Runs the validated A100 training script.
# stdout and stderr are streamed live to the notebook.
# ============================================================

import os
import subprocess
import sys


print("=" * 72)
print("MAIA LITE - SFT A100")
print("GPT-2 MEDIUM LoRA - REAL-TIME TRAINING OUTPUT")
print("=" * 72)
print()

# ------------------------------------------------------------
# Environment
# ------------------------------------------------------------

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# ------------------------------------------------------------
# Command
# ------------------------------------------------------------

command = [
    sys.executable,
    "-u",
    "-m",
    "src.train_stage4_lora_a100",
]

print("Project:")
print(f"  {SFT_PROJECT}")
print()

print("Command:")
print("  " + " ".join(command))
print()

print("Expected experiment:")
print("  Model:                  openai-community/gpt2-medium")
print("  Device:                 NVIDIA A100")
print("  Precision:              FP32")
print("  Context:                1024")
print("  Micro batch:            1")
print("  Gradient accumulation:  8")
print("  Effective batch:        8")
print("  Epochs:                 3")
print("  LoRA r/alpha/dropout:   16 / 32 / 0.05")

print()
print("=" * 72)
print("STARTING TRAINING")
print("=" * 72)
print()


# ------------------------------------------------------------
# Start process
# ------------------------------------------------------------

process = subprocess.Popen(
    command,
    cwd=str(SFT_PROJECT),
    env=env,

    # Merge stderr into stdout so traceback/warnings
    # are visible in the notebook.
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,

    # Text mode + line buffering.
    text=True,
    bufsize=1,
)


# ------------------------------------------------------------
# Stream output live
# ------------------------------------------------------------

try:
    if process.stdout is None:
        raise RuntimeError(
            "Could not capture training process output."
        )

    for line in iter(process.stdout.readline, ""):
        print(
            line,
            end="",
            flush=True,
        )

finally:
    if process.stdout is not None:
        process.stdout.close()


# ------------------------------------------------------------
# Wait for process
# ------------------------------------------------------------

return_code = process.wait()


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print()
print("=" * 72)
print(f"PROCESS EXIT CODE: {return_code}")
print("=" * 72)
print()

if return_code == 0:

    print(
        "GPT-2 MEDIUM LoRA A100: "
        "COMPLETED SUCCESSFULLY"
    )

else:

    print(
        "GPT-2 MEDIUM LoRA A100: FAILED"
    )

    raise RuntimeError(
        "Training process failed. "
        "See the complete traceback above."
    )

A saída de streaming foi truncada nas últimas 5000 linhas.
Epoch 3/3:  87%|████████▋ | 25968/30011 [1:27:31<13:51,  4.86it/s, avg=2.6342, loss=3.7447, lr=1.10e-07, update=10750/11256]
Checkpoint saved: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/a100-gpt2-medium-lora-checkpoints/checkpoint-010750

Epoch 3/3:  93%|█████████▎| 27968/30011 [1:34:17<06:57,  4.90it/s, avg=2.6342, loss=2.9898, lr=2.83e-08, update=11000/11256]
Checkpoint saved: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/a100-gpt2-medium-lora-checkpoints/checkpoint-011000

Epoch 3/3: 100%|█████████▉| 29968/30011 [1:41:02<00:08,  4.98it/s, avg=2.6339, loss=3.3893, lr=1.55e-11, update=11250/11256]
Checkpoint saved: /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/a100-gpt2-medium-lora-checkpoints/checkpoint-011250

Epoch 3/3: 100%|██████████| 30011/30011 [1:41:11<00:00,  4.94it/s, avg=2.6338, loss=2.7719, lr=0.00e+00, update=11256/11256]

Epoch 3 training mean loss: 2.633803

Val

In [ ]:
# ============================================================
# MAIA SFT VALIDATION
# GPT-2 MEDIUM + A100 LoRA
# INTERACTIVE CHAT TEST
#
# Loads:
#   - openai-community/gpt2-medium
#   - best A100 Stage 4 LoRA adapter
#
# Uses the EXACT Stage 4 prompt serialization through
# src.common_stage4.assistant_segments().
#
# Generation:
#   - greedy decoding
#   - max_new_tokens = 192
#   - no sampling
# ============================================================

import sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab_LLMs/maia-sft-validation"
)

BEST_ADAPTER_DIR = (
    PROJECT_DIR
    / "outputs"
    / "a100-gpt2-medium-lora-best"
)

BASE_MODEL_ID = "openai-community/gpt2-medium"

MAX_CONTEXT = 1024
MAX_NEW_TOKENS = 192


# ============================================================
# PROJECT IMPORTS
# ============================================================

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.common_stage4 import SYSTEM, assistant_segments


# ============================================================
# PREFLIGHT
# ============================================================

print("=" * 80)
print("GPT-2 MEDIUM + A100 LoRA - INTERACTIVE CHAT")
print("=" * 80)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not available.")

gpu_name = torch.cuda.get_device_name(0)

if "A100" not in gpu_name:
    raise RuntimeError(
        f"Expected NVIDIA A100, found: {gpu_name}"
    )

if not BEST_ADAPTER_DIR.exists():
    raise FileNotFoundError(
        f"Best LoRA adapter not found:\n{BEST_ADAPTER_DIR}"
    )

print()
print("GPU:          ", gpu_name)
print("Base model:   ", BASE_MODEL_ID)
print("LoRA adapter: ", BEST_ADAPTER_DIR)
print("System:       ", SYSTEM)
print()


# ============================================================
# TOKENIZER
# ============================================================

print("Loading tokenizer...")

try:
    tokenizer = AutoTokenizer.from_pretrained(
        BEST_ADAPTER_DIR
    )
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(
        BASE_MODEL_ID
    )

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Vocabulary:   ", len(tokenizer))
print("EOS token ID: ", tokenizer.eos_token_id)
print("PAD token ID: ", tokenizer.pad_token_id)


# ============================================================
# BASE MODEL
# ============================================================

print()
print("Loading GPT-2 Medium...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float32,
)

base_model.config.pad_token_id = tokenizer.pad_token_id


# ============================================================
# LoRA ADAPTER
# ============================================================

print("Loading A100 Stage 4 LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    BEST_ADAPTER_DIR,
    is_trainable=False,
)

model = model.to("cuda")
model.eval()

model.config.use_cache = True
model.config.pad_token_id = tokenizer.pad_token_id

print()
print("Model ready.")
print()


# ============================================================
# EXACT STAGE 4 PREFIX
# ============================================================

def build_stage4_prefix(user_text):
    """
    Build the prompt using the exact Stage 4 serialization.

    A dummy assistant response is supplied only so that
    assistant_segments() returns the prefix corresponding
    to the assistant turn. The dummy answer itself is never
    sent to the model.
    """

    messages = [
        {
            "role": "system",
            "content": SYSTEM,
        },
        {
            "role": "user",
            "content": user_text,
        },
        {
            "role": "assistant",
            "content": "__MAIA_GENERATION_PLACEHOLDER__",
        },
    ]

    segments = list(
        assistant_segments(messages)
    )

    if len(segments) != 1:
        raise RuntimeError(
            "Unexpected Stage 4 serialization result: "
            f"{len(segments)} assistant segments."
        )

    prefix, _ = segments[0]

    return prefix


# ============================================================
# GENERATION
# ============================================================

@torch.inference_mode()
def ask(user_text):
    prefix = build_stage4_prefix(user_text)

    encoded = tokenizer(
        prefix,
        add_special_tokens=False,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"]
    attention_mask = encoded["attention_mask"]

    # Reserve room for generation while respecting GPT-2's
    # 1024-token context window.
    max_prompt_tokens = (
        MAX_CONTEXT - MAX_NEW_TOKENS
    )

    if input_ids.shape[1] > max_prompt_tokens:
        input_ids = input_ids[
            :,
            -max_prompt_tokens:
        ]

        attention_mask = attention_mask[
            :,
            -max_prompt_tokens:
        ]

    input_ids = input_ids.to("cuda")
    attention_mask = attention_mask.to("cuda")

    prompt_length = input_ids.shape[1]

    generated = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,

        max_new_tokens=MAX_NEW_TOKENS,

        do_sample=False,

        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,

        use_cache=True,
    )

    new_tokens = generated[
        0,
        prompt_length:
    ]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )

    return response.strip()


# ============================================================
# SANITY CHECK: SHOW EXACT SERIALIZATION
# ============================================================

test_prompt = "What is the capital of France?"

print("=" * 80)
print("STAGE 4 SERIALIZATION CHECK")
print("=" * 80)
print()
print(build_stage4_prefix(test_prompt))
print()


# ============================================================
# QUICK MULTILINGUAL TEST
# ============================================================

tests = [
    "What is the capital of France?",
    "Explain what a neural network is in two sentences.",
    "Qual é a capital do Brasil?",
    "Explique em poucas palavras o que é uma rede neural.",
    "¿Cuál es la capital de Argentina?",
    "Explica brevemente qué es el aprendizaje automático.",
    "What is 17 + 28?",
    "Write a Python function that returns the square of a number.",
]

print("=" * 80)
print("QUICK TEST")
print("=" * 80)

for question in tests:
    print()
    print("USER:")
    print(question)

    answer = ask(question)

    print()
    print("ASSISTANT:")
    print(answer)

    print()
    print("-" * 80)


# ============================================================
# INTERACTIVE CHAT
# ============================================================

print()
print("=" * 80)
print("INTERACTIVE MODE")
print("=" * 80)
print()
print("Type a question and press Enter.")
print("Type 'exit' to finish.")
print()

while True:

    try:
        user_text = input("You: ").strip()

    except (EOFError, KeyboardInterrupt):
        print()
        print("Chat finished.")
        break

    if not user_text:
        continue

    if user_text.lower() in {
        "exit",
        "quit",
        "sair",
    }:
        print("Chat finished.")
        break

    try:
        response = ask(user_text)

        print()
        print("Assistant:")
        print(response)
        print()

    except Exception as exc:
        print()
        print(
            "Generation error:",
            repr(exc),
        )
        print()

GPT-2 MEDIUM + A100 LoRA - INTERACTIVE CHAT

GPU:           NVIDIA A100-SXM4-80GB
Base model:    openai-community/gpt2-medium
LoRA adapter:  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/outputs/a100-gpt2-medium-lora-best
System:        You are a helpful, careful multilingual assistant. Answer in the user's language unless asked otherwise.

Loading tokenizer...
Vocabulary:    50257
EOS token ID:  50256
PAD token ID:  50256

Loading GPT-2 Medium...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Loading A100 Stage 4 LoRA adapter...

Model ready.

STAGE 4 SERIALIZATION CHECK

System: You are a helpful, careful multilingual assistant. Answer in the user's language unless asked otherwise.
User: What is the capital of France?
Assistant: 

QUICK TEST

USER:
What is the capital of France?

ASSISTANT:
Paris

--------------------------------------------------------------------------------

USER:
Explain what a neural network is in two sentences.

ASSISTANT:
A neural network is a computer program that learns to process data in a way that is efficient and accurate. It is a computer program that can learn to process data in a way that is efficient and accurate.

--------------------------------------------------------------------------------

USER:
Qual é a capital do Brasil?

ASSISTANT:
A capital do Brasil é Brasil.

--------------------------------------------------------------------------------

USER:
Explique em poucas palavras o que é uma rede neural.

ASSISTANT:
"Em poucas palavras

In [1]:
# ============================================================
# MAIA LITE BASE 355M
# PRE-SFT BASELINE / COMPLETION TEST
#
# PURPOSE
# -------
# Evaluate the FINAL pretrained Maia Lite base model BEFORE
# any instruction tuning or LoRA adapter is attached.
#
# This cell:
#   1. Locates checkpoint-216567.
#   2. Locates the Maia Lite tokenizer.
#   3. Audits model/tokenizer configuration.
#   4. Loads ONLY the frozen pretrained base model.
#   5. Runs pure causal completion tests.
#   6. Runs Stage 4 assistant-format tests WITHOUT LoRA.
#
# IMPORTANT
# ---------
# This cell does NOT:
#   - create a LoRA adapter
#   - modify model weights
#   - create an optimizer
#   - perform backward()
#   - perform any training
#
# Generation is deterministic greedy decoding.
# ============================================================

from pathlib import Path
import json
import sys

import torch
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
)


# ============================================================
# GOOGLE DRIVE
# ============================================================

from google.colab import drive

print("=" * 80)
print("MOUNTING GOOGLE DRIVE")
print("=" * 80)
print()

drive.mount("/content/drive")

print()
print("Google Drive mounted.")
print()


# ============================================================
# PATHS
# ============================================================

DRIVE_ROOT = Path("/content/drive/MyDrive/Colab_LLMs")

MAIA_ROOT = DRIVE_ROOT / "Maia_Lite_v3"

SFT_PROJECT = DRIVE_ROOT / "maia-sft-validation"

CHECKPOINT_NAME = "checkpoint-216567"

MAX_NEW_TOKENS = 96


# ============================================================
# DEVICE
# ============================================================

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. "
        "This experiment is intended for the A100 runtime."
    )

device = torch.device("cuda")

gpu_name = torch.cuda.get_device_name(0)

print("=" * 80)
print("MAIA LITE BASE 355M - PRE-SFT BASELINE")
print("=" * 80)
print()
print("Device:       ", device)
print("GPU:          ", gpu_name)
print("PyTorch:      ", torch.__version__)
print("CUDA:         ", torch.version.cuda)
print("BF16 support: ", torch.cuda.is_bf16_supported())
print()


# ============================================================
# VERIFY ROOT
# ============================================================

if not MAIA_ROOT.exists():
    raise FileNotFoundError(
        f"Maia Lite root not found:\n{MAIA_ROOT}"
    )

print("Maia root:")
print(MAIA_ROOT)
print()


# ============================================================
# LOCATE FINAL CHECKPOINT
# ============================================================

print("=" * 80)
print("LOCATING FINAL CHECKPOINT")
print("=" * 80)
print()

checkpoint_candidates = sorted(
    p
    for p in MAIA_ROOT.rglob(CHECKPOINT_NAME)
    if p.is_dir()
)

if not checkpoint_candidates:
    print(
        f"No directory named {CHECKPOINT_NAME} "
        f"was found under:\n{MAIA_ROOT}"
    )
    print()
    print("Directories containing '216567':")

    alternatives = sorted(
        p
        for p in MAIA_ROOT.rglob("*216567*")
    )

    for path in alternatives[:50]:
        print(" ", path)

    raise FileNotFoundError(
        "Final checkpoint could not be located."
    )

if len(checkpoint_candidates) > 1:
    print("WARNING: multiple checkpoint candidates found:")
    for path in checkpoint_candidates:
        print(" ", path)

    raise RuntimeError(
        "Multiple checkpoint-216567 directories found. "
        "Refusing to guess which one is authoritative."
    )

checkpoint_dir = checkpoint_candidates[0]

print("Final checkpoint:")
print(checkpoint_dir)
print()


# ============================================================
# AUDIT CHECKPOINT CONTENTS
# ============================================================

print("=" * 80)
print("CHECKPOINT CONTENTS")
print("=" * 80)
print()

checkpoint_files = sorted(
    p.name
    for p in checkpoint_dir.iterdir()
)

for name in checkpoint_files:
    print(name)

print()


# ============================================================
# VERIFY MODEL FILES
# ============================================================

config_file = checkpoint_dir / "config.json"

model_candidates = [
    checkpoint_dir / "model.safetensors",
    checkpoint_dir / "pytorch_model.bin",
]

model_shards = list(
    checkpoint_dir.glob("model-*.safetensors")
)

has_model = (
    any(p.exists() for p in model_candidates)
    or bool(model_shards)
)

if not config_file.exists():
    raise FileNotFoundError(
        f"config.json not found in:\n{checkpoint_dir}"
    )

if not has_model:
    raise FileNotFoundError(
        "No Hugging Face model weights found in:\n"
        f"{checkpoint_dir}"
    )

print("Model configuration: PASS")
print("Model weights:       PASS")
print()


# ============================================================
# LOCATE MAIA TOKENIZER
# ============================================================

print("=" * 80)
print("LOCATING MAIA LITE TOKENIZER")
print("=" * 80)
print()

# Prefer tokenizer files stored directly with the final
# checkpoint. Otherwise search the Maia Lite project tree.

tokenizer_markers = [
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
]


def looks_like_tokenizer(path):
    return any(
        (path / marker).exists()
        for marker in tokenizer_markers
    )


tokenizer_candidates = []

if looks_like_tokenizer(checkpoint_dir):
    tokenizer_candidates.append(checkpoint_dir)

# Search likely tokenizer directories first.

likely_dirs = [
    MAIA_ROOT / "tokenizador",
    MAIA_ROOT / "tokenizer",
    MAIA_ROOT / "tokenizer_final",
]

for path in likely_dirs:
    if path.is_dir() and looks_like_tokenizer(path):
        tokenizer_candidates.append(path)

# General fallback search.

for marker in [
    "tokenizer.json",
    "tokenizer_config.json",
]:

    for file_path in MAIA_ROOT.rglob(marker):

        parent = file_path.parent

        if (
            looks_like_tokenizer(parent)
            and parent not in tokenizer_candidates
        ):
            tokenizer_candidates.append(parent)


print("Tokenizer candidates:")

for path in tokenizer_candidates:
    print(" ", path)

print()

if not tokenizer_candidates:
    raise FileNotFoundError(
        "No Maia Lite tokenizer could be located."
    )


# ============================================================
# IDENTIFY TOKENIZER BY EXPECTED MAIA IDENTITY
# ============================================================

print("=" * 80)
print("TOKENIZER AUDIT")
print("=" * 80)
print()

valid_tokenizers = []

for path in tokenizer_candidates:

    try:
        tok = AutoTokenizer.from_pretrained(
            path,
            local_files_only=True,
        )

        info = {
            "path": path,
            "vocab": len(tok),
            "bos": tok.bos_token_id,
            "pad": tok.pad_token_id,
            "eos": tok.eos_token_id,
            "unk": tok.unk_token_id,
            "mask": tok.mask_token_id,
        }

        print("Candidate:")
        print(" ", path)
        print("  vocab:", info["vocab"])
        print("  BOS:  ", info["bos"])
        print("  PAD:  ", info["pad"])
        print("  EOS:  ", info["eos"])
        print("  UNK:  ", info["unk"])
        print("  MASK: ", info["mask"])
        print()

        # Frozen Maia Lite tokenizer identity:
        #
        # vocab = 50257
        # BOS   = 0
        # PAD   = 1
        # EOS   = 2
        # UNK   = 3
        # MASK  = 4

        if (
            len(tok) == 50257
            and tok.bos_token_id == 0
            and tok.pad_token_id == 1
            and tok.eos_token_id == 2
            and tok.unk_token_id == 3
            and tok.mask_token_id == 4
        ):
            valid_tokenizers.append(
                (path, tok)
            )

    except Exception as exc:
        print(
            "Could not load tokenizer candidate:",
            path,
        )
        print(repr(exc))
        print()


if not valid_tokenizers:
    raise RuntimeError(
        "No tokenizer matched the frozen Maia Lite "
        "tokenizer identity."
    )

# Multiple directories are acceptable only if they encode
# the same tokenizer. Prefer the checkpoint copy if present.

selected = None

for path, tok in valid_tokenizers:
    if path == checkpoint_dir:
        selected = (path, tok)
        break

if selected is None:
    selected = valid_tokenizers[0]

tokenizer_dir, tokenizer = selected

print("Selected tokenizer:")
print(tokenizer_dir)
print()

print("Tokenizer identity:")
print("  Vocabulary:", len(tokenizer))
print("  BOS:       ", tokenizer.bos_token_id)
print("  PAD:       ", tokenizer.pad_token_id)
print("  EOS:       ", tokenizer.eos_token_id)
print("  UNK:       ", tokenizer.unk_token_id)
print("  MASK:      ", tokenizer.mask_token_id)
print()


# ============================================================
# MODEL CONFIGURATION AUDIT
# ============================================================

print("=" * 80)
print("MODEL CONFIGURATION")
print("=" * 80)
print()

config = AutoConfig.from_pretrained(
    checkpoint_dir,
    local_files_only=True,
)

print("Architecture:   ", config.architectures)
print("Model type:     ", config.model_type)
print("Vocabulary:     ", config.vocab_size)
print("Context:        ", config.n_positions)
print("Embedding size: ", config.n_embd)
print("Layers:         ", config.n_layer)
print("Heads:          ", config.n_head)
print("BOS:            ", config.bos_token_id)
print("EOS:            ", config.eos_token_id)
print("PAD:            ", config.pad_token_id)
print()


# ============================================================
# STRICT CONFIG CHECK
# ============================================================

expected_config = {
    "vocab_size": 50257,
    "n_positions": 1024,
    "n_embd": 1024,
    "n_layer": 24,
    "n_head": 16,
    "bos_token_id": 0,
    "eos_token_id": 2,
    "pad_token_id": 1,
}

for key, expected in expected_config.items():

    actual = getattr(
        config,
        key,
        None,
    )

    if actual != expected:
        raise RuntimeError(
            f"Unexpected Maia configuration: "
            f"{key}={actual}, expected={expected}"
        )

print("Maia Lite architecture identity: PASS")
print()


# ============================================================
# LOAD FROZEN BASE MODEL
# ============================================================

print("=" * 80)
print("LOADING FROZEN MAIA LITE BASE")
print("=" * 80)
print()

model = AutoModelForCausalLM.from_pretrained(
    checkpoint_dir,
    local_files_only=True,
    dtype=torch.float32,
)

model = model.to(device)

model.eval()

model.config.use_cache = True
model.config.pad_token_id = tokenizer.pad_token_id

for parameter in model.parameters():
    parameter.requires_grad_(False)

parameter_count = sum(
    p.numel()
    for p in model.parameters()
)

trainable_count = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Parameters:           {parameter_count:,}")
print(f"Trainable parameters: {trainable_count:,}")
print()

if parameter_count != 354_823_168:
    raise RuntimeError(
        "Unexpected parameter count. "
        f"Found {parameter_count:,}; "
        "expected 354,823,168."
    )

if trainable_count != 0:
    raise RuntimeError(
        "Base model is not fully frozen."
    )

print("Frozen Maia Lite Base: PASS")
print()


# ============================================================
# GENERATION
# ============================================================

@torch.inference_mode()
def complete(
    prompt,
    max_new_tokens=MAX_NEW_TOKENS,
):

    encoded = tokenizer(
        prompt,
        add_special_tokens=False,
        return_tensors="pt",
    )

    input_ids = encoded["input_ids"].to(device)

    attention_mask = encoded[
        "attention_mask"
    ].to(device)

    prompt_length = input_ids.shape[1]

    if prompt_length >= config.n_positions:
        raise RuntimeError(
            f"Prompt is too long: "
            f"{prompt_length} tokens."
        )

    available = (
        config.n_positions
        - prompt_length
    )

    generation_length = min(
        max_new_tokens,
        available,
    )

    generated = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=generation_length,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )

    new_tokens = generated[
        0,
        prompt_length:
    ]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    )


# ============================================================
# TEST 1
# PURE PRETRAINING COMPLETION
# ============================================================

pure_completions = [
    "The capital of France is",
    "The capital of Brazil is",
    "The capital of the United States is",
    "A neural network is",
    "A transistor is",
    "A black hole is",
    "17 + 28 =",
    "def square(x):",
    "A capital do Brasil é",
    "Uma rede neural é",
    "Um autômato celular é",
    "La capital de Argentina es",
    "El aprendizaje automático es",
]


print("=" * 80)
print("TEST 1 - PURE CAUSAL COMPLETION")
print("=" * 80)

for prompt in pure_completions:

    print()
    print("PROMPT:")
    print(repr(prompt))

    response = complete(
        prompt,
        max_new_tokens=64,
    )

    print()
    print("COMPLETION:")
    print(response)

    print()
    print("-" * 80)


# ============================================================
# TEST 2
# STAGE 4 ASSISTANT FORMAT WITHOUT SFT
# ============================================================

SYSTEM = (
    "You are a helpful, careful multilingual assistant. "
    "Answer in the user's language unless asked otherwise."
)


def assistant_prompt(user_text):

    return (
        f"System: {SYSTEM}\n"
        f"User: {user_text}\n"
        f"Assistant: "
    )


assistant_tests = [
    "What is the capital of France?",
    "Explain what a neural network is in two sentences.",
    "Qual é a capital do Brasil?",
    "Explique em poucas palavras o que é uma rede neural.",
    "¿Cuál es la capital de Argentina?",
    "Explica brevemente qué es el aprendizaje automático.",
    "What is 17 + 28?",
    "Write a Python function that returns the square of a number.",
    "O que é um autômato celular?",
    "What is a transistor?",
    "What is an automata?",
    "What is a black hole?",
    "What is the capital of Brazil?",
    "What is the capital of United States of America?",
    "Who are you?",
]


print()
print("=" * 80)
print("TEST 2 - ASSISTANT FORMAT BEFORE SFT")
print("=" * 80)

for question in assistant_tests:

    prompt = assistant_prompt(
        question
    )

    print()
    print("USER:")
    print(question)

    response = complete(
        prompt,
        max_new_tokens=192,
    )

    print()
    print("BASE MODEL:")
    print(response)

    print()
    print("-" * 80)


# ============================================================
# INTERACTIVE PURE COMPLETION
# ============================================================

print()
print("=" * 80)
print("INTERACTIVE PURE COMPLETION")
print("=" * 80)
print()
print(
    "Enter a raw text prefix for the Maia Lite Base."
)
print(
    "Example: A capital do Brasil é"
)
print(
    "Type 'exit' to finish."
)
print()

while True:

    try:
        prompt = input("Prefix: ").strip()

    except (EOFError, KeyboardInterrupt):
        print()
        break

    if not prompt:
        continue

    if prompt.lower() in {
        "exit",
        "quit",
        "sair",
    }:
        break

    response = complete(
        prompt,
        max_new_tokens=96,
    )

    print()
    print("Completion:")
    print(response)
    print()


print()
print("=" * 80)
print("PRE-SFT BASELINE COMPLETE")
print("=" * 80)

MOUNTING GOOGLE DRIVE

Mounted at /content/drive

Google Drive mounted.

MAIA LITE BASE 355M - PRE-SFT BASELINE

Device:        cuda
GPU:           NVIDIA A100-SXM4-80GB
PyTorch:       2.11.0+cu130
CUDA:          13.0
BF16 support:  True

Maia root:
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3

LOCATING FINAL CHECKPOINT

Final checkpoint:
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/checkpoints/checkpoint-216567

CHECKPOINT CONTENTS

config.json
generation_config.json
model.safetensors
optimizer.pt
rng_state.pth
scheduler.pt
trainer_state.json
training_args.bin

Model configuration: PASS
Model weights:       PASS

LOCATING MAIA LITE TOKENIZER

Tokenizer candidates:
  /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer

TOKENIZER AUDIT

Candidate:
  /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer
  vocab: 50257
  BOS:   0
  PAD:   1
  EOS:   2
  UNK:   3
  MASK:  4

Selected tokenizer:
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer

Tokenizer identity:
  Voca

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Parameters:           354,823,168
Trainable parameters: 0

Frozen Maia Lite Base: PASS

TEST 1 - PURE CAUSAL COMPLETION

PROMPT:
'The capital of France is'

COMPLETION:
 located in the heart of the country, and it is the capital of the country. The capital of France is located in the heart of the country, and it is the capital of the country. The capital of France is located in the heart of the country, and it is the capital of the country.
The capital

--------------------------------------------------------------------------------

PROMPT:
'The capital of Brazil is'

COMPLETION:
 located in the state of São Paulo, and it is the largest city in the country. The city is known for its beautiful architecture, and it is also known for its rich history and culture. The city is also known for its rich history and culture, and it is also known for its rich history and culture.


--------------------------------------------------------------------------------

PROMPT:
'The capital of the Unit

In [2]:
# ============================================================
# MAIA LITE 355M + LoRA
# A100 STAGE 4 CONTROL
#
# This cell CREATES the training script.
# It DOES NOT start training.
# ============================================================

from pathlib import Path
import textwrap
import py_compile


PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab_LLMs/maia-sft-validation"
)

SCRIPT = (
    PROJECT_DIR
    / "src"
    / "train_stage4_maia_lora_a100.py"
)


code = r'''
#!/usr/bin/env python3

# ============================================================
# MAIA LITE 355M + LoRA
# STAGE 4 - A100 CONTROL
#
# Base:
#   Maia Lite Base 355M
#   checkpoint-216567
#
# Objective:
#   Response-only causal language modeling
#
# Experimental control:
#   Same Stage 4 corpus and LoRA configuration used for
#   GPT-2 Medium + LoRA A100.
#
# IMPORTANT:
#   The pretrained Maia Lite checkpoint is never modified.
# ============================================================

import gc
import json
import math
import random
import shutil
import sys
from pathlib import Path

import torch

from torch.optim import AdamW
from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab_LLMs/maia-sft-validation"
)

MAIA_ROOT = Path(
    "/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3"
)

MAIA_CHECKPOINT = (
    MAIA_ROOT
    / "checkpoints"
    / "checkpoint-216567"
)

MAIA_TOKENIZER = (
    MAIA_ROOT
    / "tokenizer"
)

DATA_DIR = (
    PROJECT_DIR
    / "data"
    / "stage4"
)

TRAIN_FILE = DATA_DIR / "train.jsonl"
VALIDATION_FILE = DATA_DIR / "validation.jsonl"

OUTPUT_ROOT = PROJECT_DIR / "outputs"

CHECKPOINT_DIR = (
    OUTPUT_ROOT
    / "a100-maia-lite-lora-checkpoints"
)

BEST_DIR = (
    OUTPUT_ROOT
    / "a100-maia-lite-lora-best"
)

FINAL_DIR = (
    OUTPUT_ROOT
    / "a100-maia-lite-lora-final"
)

METRICS_FILE = (
    OUTPUT_ROOT
    / "a100-maia-lite-lora-training-metrics.json"
)


# ============================================================
# FROZEN EXPERIMENT CONFIGURATION
# ============================================================

SEED = 42

MAX_LENGTH = 1024

MICRO_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

EPOCHS = 3

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

CHECKPOINT_INTERVAL = 250

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

LORA_TARGET_MODULES = [
    "c_attn",
    "c_proj",
    "c_fc",
]


# ============================================================
# IMPORT PROJECT DATASET
# ============================================================

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_DIR),
    )

from src.dataset_stage4 import ResponseOnlyDataset


# ============================================================
# UTILITIES
# ============================================================

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def read_jsonl(path):
    rows = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return rows


def collate_batch(batch):
    max_len = max(
        len(item["input_ids"])
        for item in batch
    )

    input_ids = []
    labels = []
    attention_mask = []

    for item in batch:
        ids = item["input_ids"]
        lab = item["labels"]

        padding = max_len - len(ids)

        input_ids.append(
            ids + [1] * padding
        )

        labels.append(
            lab + [-100] * padding
        )

        attention_mask.append(
            [1] * len(ids)
            + [0] * padding
        )

    return {
        "input_ids": torch.tensor(
            input_ids,
            dtype=torch.long,
        ),
        "labels": torch.tensor(
            labels,
            dtype=torch.long,
        ),
        "attention_mask": torch.tensor(
            attention_mask,
            dtype=torch.long,
        ),
    }


@torch.no_grad()
def evaluate(
    model,
    loader,
    device,
    description,
):
    model.eval()

    loss_sum = 0.0
    batches = 0

    progress = tqdm(
        loader,
        desc=description,
    )

    for batch in progress:
        batch = {
            key: value.to(
                device,
                non_blocking=True,
            )
            for key, value
            in batch.items()
        }

        outputs = model(
            **batch
        )

        loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Non-finite validation loss."
            )

        value = loss.item()

        loss_sum += value
        batches += 1

        progress.set_postfix(
            loss=f"{value:.4f}",
            avg=f"{loss_sum / batches:.4f}",
        )

        del outputs
        del loss
        del batch

    progress.close()

    model.train()

    return loss_sum / max(
        1,
        batches,
    )


def write_metrics(
    initial_validation_loss,
    best_validation_loss,
    global_update,
    total_updates,
    history,
    status,
):
    payload = {
        "experiment":
            "maia-lite-355m-stage4-lora-a100",

        "base_checkpoint":
            str(MAIA_CHECKPOINT),

        "tokenizer":
            str(MAIA_TOKENIZER),

        "status":
            status,

        "initial_validation_loss":
            initial_validation_loss,

        "best_validation_loss":
            best_validation_loss,

        "optimizer_updates":
            global_update,

        "total_optimizer_updates":
            total_updates,

        "history":
            history,

        "lora_config": {
            "r":
                LORA_R,

            "lora_alpha":
                LORA_ALPHA,

            "lora_dropout":
                LORA_DROPOUT,

            "target_modules":
                LORA_TARGET_MODULES,

            "bias":
                "none",

            "task_type":
                "CAUSAL_LM",
        },
    }

    METRICS_FILE.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    with METRICS_FILE.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            payload,
            f,
            indent=2,
        )


def save_checkpoint(
    model,
    tokenizer,
    optimizer,
    scheduler,
    epoch,
    batch_in_epoch,
    global_update,
    total_updates,
    best_validation_loss,
    initial_validation_loss,
    history,
):
    path = (
        CHECKPOINT_DIR
        / f"checkpoint-{global_update:06d}"
    )

    path.mkdir(
        parents=True,
        exist_ok=True,
    )

    model.save_pretrained(
        path,
        safe_serialization=True,
    )

    tokenizer.save_pretrained(
        path
    )

    state = {
        "epoch":
            epoch,

        "batch_in_epoch":
            batch_in_epoch,

        "global_update":
            global_update,

        "total_updates":
            total_updates,

        "best_validation_loss":
            best_validation_loss,

        "initial_validation_loss":
            initial_validation_loss,

        "history":
            history,

        "optimizer":
            optimizer.state_dict(),

        "scheduler":
            scheduler.state_dict(),

        "torch_rng_state":
            torch.get_rng_state(),

        "cuda_rng_state":
            torch.cuda.get_rng_state_all(),
    }

    torch.save(
        state,
        path / "training_state.pt",
    )

    return path


# ============================================================
# SEED
# ============================================================

seed_everything(SEED)


# ============================================================
# A100 PREFLIGHT
# ============================================================

print("=" * 80)
print("MAIA LITE 355M + LoRA - A100 STAGE 4 CONTROL")
print("=" * 80)
print()

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required."
    )

device = torch.device("cuda")

gpu_name = torch.cuda.get_device_name(0)

gpu_memory = (
    torch.cuda
    .get_device_properties(0)
    .total_memory
    / 1024**3
)

print("GPU:                 ", gpu_name)
print(
    "GPU memory:          ",
    f"{gpu_memory:.2f} GiB",
)
print("PyTorch:             ", torch.__version__)
print("CUDA:                ", torch.version.cuda)
print(
    "BF16 support:        ",
    torch.cuda.is_bf16_supported(),
)
print()

if "A100" not in gpu_name:
    raise RuntimeError(
        "This experiment requires an NVIDIA A100."
    )


# ============================================================
# FILE AUDIT
# ============================================================

required = [
    MAIA_CHECKPOINT,
    MAIA_CHECKPOINT / "config.json",
    MAIA_CHECKPOINT / "model.safetensors",
    MAIA_TOKENIZER,
    TRAIN_FILE,
    VALIDATION_FILE,
]

for path in required:
    if not path.exists():
        raise FileNotFoundError(
            f"Required path not found:\n{path}"
        )

print("Required files: PASS")
print()


# ============================================================
# TOKENIZER AUDIT
# ============================================================

print("=" * 80)
print("LOADING MAIA TOKENIZER")
print("=" * 80)
print()

tokenizer = AutoTokenizer.from_pretrained(
    MAIA_TOKENIZER,
    local_files_only=True,
)

expected_tokenizer = {
    "vocab": 50257,
    "bos": 0,
    "pad": 1,
    "eos": 2,
    "unk": 3,
    "mask": 4,
}

actual_tokenizer = {
    "vocab":
        len(tokenizer),

    "bos":
        tokenizer.bos_token_id,

    "pad":
        tokenizer.pad_token_id,

    "eos":
        tokenizer.eos_token_id,

    "unk":
        tokenizer.unk_token_id,

    "mask":
        tokenizer.mask_token_id,
}

print(
    json.dumps(
        actual_tokenizer,
        indent=2,
    )
)
print()

if actual_tokenizer != expected_tokenizer:
    raise RuntimeError(
        "Maia tokenizer identity mismatch."
    )

print("Maia tokenizer identity: PASS")
print()


# ============================================================
# MODEL CONFIG AUDIT
# ============================================================

print("=" * 80)
print("AUDITING MAIA MODEL CONFIGURATION")
print("=" * 80)
print()

config = AutoConfig.from_pretrained(
    MAIA_CHECKPOINT,
    local_files_only=True,
)

expected_model = {
    "vocab_size": 50257,
    "n_positions": 1024,
    "n_embd": 1024,
    "n_layer": 24,
    "n_head": 16,
    "bos_token_id": 0,
    "eos_token_id": 2,
    "pad_token_id": 1,
}

for key, expected in expected_model.items():
    actual = getattr(
        config,
        key,
        None,
    )

    print(
        f"{key:16s}: "
        f"{actual}"
    )

    if actual != expected:
        raise RuntimeError(
            f"Model identity mismatch: "
            f"{key}={actual}, "
            f"expected={expected}"
        )

print()
print("Maia architecture identity: PASS")
print()


# ============================================================
# LOAD MAIA LITE BASE
# ============================================================

print("=" * 80)
print("LOADING MAIA LITE BASE")
print("=" * 80)
print()

base_model = (
    AutoModelForCausalLM
    .from_pretrained(
        MAIA_CHECKPOINT,
        local_files_only=True,
        dtype=torch.float32,
    )
)

base_parameters = sum(
    p.numel()
    for p in base_model.parameters()
)

print(
    f"Base parameters:      "
    f"{base_parameters:,}"
)

if base_parameters != 354_823_168:
    raise RuntimeError(
        "Unexpected Maia Lite parameter count."
    )

base_model.config.use_cache = False

print("Base model identity: PASS")
print()


# ============================================================
# ATTACH LoRA
# ============================================================

print("=" * 80)
print("ATTACHING LoRA")
print("=" * 80)
print()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGET_MODULES,
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(
    base_model,
    lora_config,
)

model.config.use_cache = False

model.gradient_checkpointing_enable()

model.enable_input_require_grads()

model = model.to(device)

total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

trainable_percentage = (
    100.0
    * trainable_parameters
    / total_parameters
)

print("LoRA configuration:")
print("  rank:                  ", LORA_R)
print("  alpha:                 ", LORA_ALPHA)
print("  dropout:               ", LORA_DROPOUT)
print(
    "  target modules:        ",
    ", ".join(
        LORA_TARGET_MODULES
    ),
)
print("  bias:                   none")
print(
    "  use_cache:             ",
    model.config.use_cache,
)
print(
    "  gradient checkpointing: True"
)
print(
    f"  total parameters:       "
    f"{total_parameters:,}"
)
print(
    f"  trainable parameters:   "
    f"{trainable_parameters:,}"
)
print(
    f"  trainable percentage:   "
    f"{trainable_percentage:.4f}%"
)
print()


# ============================================================
# LOAD FROZEN STAGE 4 DATA
# ============================================================

print("=" * 80)
print("LOADING FROZEN STAGE 4 DATA")
print("=" * 80)
print()

train_rows = read_jsonl(
    TRAIN_FILE
)

validation_rows = read_jsonl(
    VALIDATION_FILE
)

train_dataset = ResponseOnlyDataset(
    train_rows,
    tokenizer,
    MAX_LENGTH,
)

validation_dataset = ResponseOnlyDataset(
    validation_rows,
    tokenizer,
    MAX_LENGTH,
)

print(
    f"Training conversations:   "
    f"{len(train_rows):,}"
)
print(
    f"Validation conversations: "
    f"{len(validation_rows):,}"
)
print(
    f"Training segments:        "
    f"{len(train_dataset):,}"
)
print(
    f"Validation segments:      "
    f"{len(validation_dataset):,}"
)
print()


# ============================================================
# DATALOADERS
# ============================================================

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=MICRO_BATCH_SIZE,
    shuffle=True,
    generator=generator,
    collate_fn=collate_batch,
    num_workers=0,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=0,
)


# ============================================================
# TRAINING BUDGET
# ============================================================

batches_per_epoch = len(
    train_loader
)

updates_per_epoch = (
    batches_per_epoch
    // GRADIENT_ACCUMULATION_STEPS
)

total_updates = (
    updates_per_epoch
    * EPOCHS
)

warmup_updates = round(
    total_updates
    * WARMUP_RATIO
)

print("=" * 80)
print("TRAINING CONFIGURATION")
print("=" * 80)
print()

print("Precision:              FP32")
print("Max length:             ", MAX_LENGTH)
print("Micro batch size:       ", MICRO_BATCH_SIZE)
print(
    "Gradient accumulation: ",
    GRADIENT_ACCUMULATION_STEPS,
)
print(
    "Effective batch size:   ",
    MICRO_BATCH_SIZE
    * GRADIENT_ACCUMULATION_STEPS,
)
print("Epochs:                 ", EPOCHS)
print(
    f"Batches/epoch:          "
    f"{batches_per_epoch:,}"
)
print(
    f"Updates/epoch:          "
    f"{updates_per_epoch:,}"
)
print(
    f"Total updates:          "
    f"{total_updates:,}"
)
print(
    f"Warmup updates:         "
    f"{warmup_updates:,}"
)
print(
    f"Learning rate:          "
    f"{LEARNING_RATE}"
)
print(
    f"Weight decay:           "
    f"{WEIGHT_DECAY}"
)
print()


# ============================================================
# OPTIMIZER / SCHEDULER
# ============================================================

trainable = [
    p
    for p in model.parameters()
    if p.requires_grad
]

optimizer = AdamW(
    trainable,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    foreach=False,
)

scheduler = (
    get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_updates,
        num_training_steps=total_updates,
    )
)


# ============================================================
# INITIAL VALIDATION
# ============================================================

print("=" * 80)
print("INITIAL VALIDATION")
print("=" * 80)
print()

initial_validation_loss = evaluate(
    model,
    validation_loader,
    device,
    "Initial validation",
)

print()
print(
    f"Initial validation loss: "
    f"{initial_validation_loss:.12f}"
)
print()


# ============================================================
# TRAINING
# ============================================================

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

best_validation_loss = float(
    "inf"
)

global_update = 0

history = []

optimizer.zero_grad(
    set_to_none=True
)


for epoch in range(
    1,
    EPOCHS + 1,
):
    print()
    print("=" * 80)
    print(
        f"EPOCH {epoch}/{EPOCHS}"
    )
    print("=" * 80)

    model.train()

    epoch_loss_sum = 0.0
    epoch_loss_batches = 0

    progress = tqdm(
        enumerate(
            train_loader,
            start=1,
        ),
        total=len(train_loader),
        desc=f"Epoch {epoch}/{EPOCHS}",
    )

    for batch_index, batch in progress:
        batch = {
            key: value.to(
                device,
                non_blocking=True,
            )
            for key, value
            in batch.items()
        }

        outputs = model(
            **batch
        )

        loss = outputs.loss

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at "
                f"epoch={epoch}, "
                f"batch={batch_index}, "
                f"update={global_update}"
            )

        raw_loss = loss.detach().item()

        epoch_loss_sum += raw_loss
        epoch_loss_batches += 1

        scaled_loss = (
            loss
            / GRADIENT_ACCUMULATION_STEPS
        )

        scaled_loss.backward()

        del scaled_loss
        del loss
        del outputs
        del batch

        if (
            batch_index
            % GRADIENT_ACCUMULATION_STEPS
            == 0
        ):
            torch.nn.utils.clip_grad_norm_(
                trainable,
                MAX_GRAD_NORM,
            )

            optimizer.step()
            scheduler.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            global_update += 1

            current_lr = (
                scheduler
                .get_last_lr()[0]
            )

            progress.set_postfix(
                loss=f"{raw_loss:.4f}",
                avg=(
                    f"{epoch_loss_sum / epoch_loss_batches:.4f}"
                ),
                lr=f"{current_lr:.2e}",
            )

            if (
                global_update
                % CHECKPOINT_INTERVAL
                == 0
            ):
                checkpoint = save_checkpoint(
                    model=model,
                    tokenizer=tokenizer,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    epoch=epoch,
                    batch_in_epoch=batch_index,
                    global_update=global_update,
                    total_updates=total_updates,
                    best_validation_loss=best_validation_loss,
                    initial_validation_loss=initial_validation_loss,
                    history=history,
                )

                print()
                print(
                    "Checkpoint saved:",
                    checkpoint,
                )

                write_metrics(
                    initial_validation_loss,
                    best_validation_loss,
                    global_update,
                    total_updates,
                    history,
                    status="training",
                )

    progress.close()

    epoch_training_loss = (
        epoch_loss_sum
        / max(
            1,
            epoch_loss_batches,
        )
    )

    print()
    print(
        f"Epoch {epoch} training mean loss: "
        f"{epoch_training_loss:.12f}"
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    validation_loss = evaluate(
        model,
        validation_loader,
        device,
        f"Validation {epoch}/{EPOCHS}",
    )

    epoch_record = {
        "epoch":
            epoch,

        "update":
            global_update,

        "training_loss":
            epoch_training_loss,

        "validation_loss":
            validation_loss,

        "learning_rate":
            scheduler.get_last_lr()[0],
    }

    history.append(
        epoch_record
    )

    print()
    print(
        json.dumps(
            epoch_record,
            indent=2,
        )
    )


    # ========================================================
    # BEST MODEL
    # ========================================================

    if (
        validation_loss
        < best_validation_loss
    ):
        best_validation_loss = (
            validation_loss
        )

        print()
        print(
            "New best validation loss:",
            f"{best_validation_loss:.12f}",
        )

        if BEST_DIR.exists():
            shutil.rmtree(
                BEST_DIR
            )

        BEST_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        model.save_pretrained(
            BEST_DIR,
            safe_serialization=True,
        )

        tokenizer.save_pretrained(
            BEST_DIR
        )


    # ========================================================
    # EPOCH-END CHECKPOINT
    # ========================================================

    checkpoint = save_checkpoint(
        model=model,
        tokenizer=tokenizer,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        batch_in_epoch=len(train_loader),
        global_update=global_update,
        total_updates=total_updates,
        best_validation_loss=best_validation_loss,
        initial_validation_loss=initial_validation_loss,
        history=history,
    )

    print()
    print(
        "Checkpoint saved:",
        checkpoint,
    )

    write_metrics(
        initial_validation_loss,
        best_validation_loss,
        global_update,
        total_updates,
        history,
        status="training",
    )


# ============================================================
# SAVE FINAL ADAPTER
# ============================================================

print()
print("=" * 80)
print("SAVING FINAL MODEL")
print("=" * 80)
print()

if FINAL_DIR.exists():
    shutil.rmtree(
        FINAL_DIR
    )

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

model.save_pretrained(
    FINAL_DIR,
    safe_serialization=True,
)

tokenizer.save_pretrained(
    FINAL_DIR
)


# ============================================================
# FINAL METRICS
# ============================================================

write_metrics(
    initial_validation_loss,
    best_validation_loss,
    global_update,
    total_updates,
    history,
    status="complete",
)

final_metrics = {
    "initial_validation_loss":
        initial_validation_loss,

    "best_validation_loss":
        best_validation_loss,

    "optimizer_updates":
        global_update,

    "total_optimizer_updates":
        total_updates,

    "history":
        history,
}

print()
print("=" * 80)
print("MAIA LITE LoRA TRAINING COMPLETE")
print("=" * 80)
print()

print(
    json.dumps(
        final_metrics,
        indent=2,
    )
)

print()
print(
    "Best model:",
    BEST_DIR,
)
print(
    "Final model:",
    FINAL_DIR,
)
print(
    "Metrics:",
    METRICS_FILE,
)
'''


# ============================================================
# WRITE SCRIPT
# ============================================================

SCRIPT.parent.mkdir(
    parents=True,
    exist_ok=True,
)

SCRIPT.write_text(
    textwrap.dedent(
        code
    ).lstrip(),
    encoding="utf-8",
)

print("=" * 80)
print("MAIA LITE LoRA A100 SCRIPT CREATED")
print("=" * 80)
print()
print("Script:")
print(SCRIPT)
print()


# ============================================================
# SYNTAX CHECK
# ============================================================

py_compile.compile(
    str(SCRIPT),
    doraise=True,
)

print("Python syntax: PASS")
print()


# ============================================================
# STRUCTURAL AUDIT
# ============================================================

generated = SCRIPT.read_text(
    encoding="utf-8"
)

required_markers = [
    "checkpoint-216567",
    "a100-maia-lite-lora-checkpoints",
    "a100-maia-lite-lora-best",
    "a100-maia-lite-lora-final",
    "a100-maia-lite-lora-training-metrics.json",
    "LORA_R = 16",
    "LORA_ALPHA = 32",
    "LORA_DROPOUT = 0.05",
    '"c_attn"',
    '"c_proj"',
    '"c_fc"',
    "MICRO_BATCH_SIZE = 1",
    "GRADIENT_ACCUMULATION_STEPS = 8",
    "EPOCHS = 3",
    "LEARNING_RATE = 2e-5",
]

for marker in required_markers:
    if marker not in generated:
        raise RuntimeError(
            f"Missing required marker: "
            f"{marker}"
        )

print("Structural audit: PASS")
print()

print(
    "The training script has been created, "
    "but training has NOT started."
)

MAIA LITE LoRA A100 SCRIPT CREATED

Script:
/content/drive/MyDrive/Colab_LLMs/maia-sft-validation/src/train_stage4_maia_lora_a100.py

Python syntax: PASS

Structural audit: PASS

The training script has been created, but training has NOT started.


In [3]:
# ============================================================
# MAIA LITE STAGE 4
# TOKENIZATION / DATASET PRE-TRAINING AUDIT
#
# No model is loaded.
# No LoRA is created.
# No training is performed.
# ============================================================

from pathlib import Path
import json
import sys

from transformers import AutoTokenizer


PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab_LLMs/maia-sft-validation"
)

MAIA_TOKENIZER = Path(
    "/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer"
)

DATA_DIR = PROJECT_DIR / "data" / "stage4"

TRAIN_FILE = DATA_DIR / "train.jsonl"
VALIDATION_FILE = DATA_DIR / "validation.jsonl"
TEST_FILE = DATA_DIR / "test.jsonl"

MAX_LENGTH = 1024


# ============================================================
# IMPORT PROJECT DATASET
# ============================================================

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_DIR),
    )

from src.dataset_stage4 import ResponseOnlyDataset


# ============================================================
# UTILITIES
# ============================================================

def read_jsonl(path):
    rows = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return rows


# ============================================================
# TOKENIZER
# ============================================================

print("=" * 80)
print("MAIA LITE STAGE 4 - PRE-TRAINING DATASET AUDIT")
print("=" * 80)
print()

tokenizer = AutoTokenizer.from_pretrained(
    MAIA_TOKENIZER,
    local_files_only=True,
)

print("Tokenizer:")
print(MAIA_TOKENIZER)
print()
print("Vocabulary: ", len(tokenizer))
print("BOS:        ", tokenizer.bos_token_id)
print("PAD:        ", tokenizer.pad_token_id)
print("EOS:        ", tokenizer.eos_token_id)
print("UNK:        ", tokenizer.unk_token_id)
print("MASK:       ", tokenizer.mask_token_id)
print()


expected = (
    len(tokenizer) == 50257
    and tokenizer.bos_token_id == 0
    and tokenizer.pad_token_id == 1
    and tokenizer.eos_token_id == 2
    and tokenizer.unk_token_id == 3
    and tokenizer.mask_token_id == 4
)

if not expected:
    raise RuntimeError(
        "Maia Lite tokenizer identity mismatch."
    )

print("Tokenizer identity: PASS")
print()


# ============================================================
# LOAD FROZEN SPLITS
# ============================================================

train_rows = read_jsonl(
    TRAIN_FILE
)

validation_rows = read_jsonl(
    VALIDATION_FILE
)

test_rows = read_jsonl(
    TEST_FILE
)

print("Frozen conversations:")
print(f"  TRAIN: {len(train_rows):,}")
print(f"  VAL:   {len(validation_rows):,}")
print(f"  TEST:  {len(test_rows):,}")
print()


# ============================================================
# MATERIALIZE RESPONSE-ONLY DATASETS
# ============================================================

print("Tokenizing TRAIN with Maia tokenizer...")

train_dataset = ResponseOnlyDataset(
    train_rows,
    tokenizer,
    MAX_LENGTH,
)

print("Tokenizing VALIDATION with Maia tokenizer...")

validation_dataset = ResponseOnlyDataset(
    validation_rows,
    tokenizer,
    MAX_LENGTH,
)

print("Tokenizing TEST with Maia tokenizer...")

test_dataset = ResponseOnlyDataset(
    test_rows,
    tokenizer,
    MAX_LENGTH,
)

print()


# ============================================================
# RESULTS
# ============================================================

print("=" * 80)
print("MAIA TOKENIZATION RESULTS")
print("=" * 80)
print()

print(
    f"TRAIN conversations:     "
    f"{len(train_rows):,}"
)

print(
    f"TRAIN accepted segments: "
    f"{len(train_dataset):,}"
)

print()

print(
    f"VAL conversations:       "
    f"{len(validation_rows):,}"
)

print(
    f"VAL accepted segments:   "
    f"{len(validation_dataset):,}"
)

print()

print(
    f"TEST conversations:      "
    f"{len(test_rows):,}"
)

print(
    f"TEST accepted segments:  "
    f"{len(test_dataset):,}"
)

print()


# ============================================================
# COMPARE WITH GPT-2 CONTROL
# ============================================================

GPT2_TRAIN_SEGMENTS = 30011
GPT2_VAL_SEGMENTS = 1721
GPT2_TEST_SEGMENTS = 1691

print("=" * 80)
print("COMPARISON WITH GPT-2 MEDIUM STAGE 4")
print("=" * 80)
print()

print(
    "TRAIN:"
    f" GPT-2={GPT2_TRAIN_SEGMENTS:,}"
    f" | Maia={len(train_dataset):,}"
    f" | delta={len(train_dataset) - GPT2_TRAIN_SEGMENTS:+,}"
)

print(
    "VAL:  "
    f" GPT-2={GPT2_VAL_SEGMENTS:,}"
    f" | Maia={len(validation_dataset):,}"
    f" | delta={len(validation_dataset) - GPT2_VAL_SEGMENTS:+,}"
)

print(
    "TEST: "
    f" GPT-2={GPT2_TEST_SEGMENTS:,}"
    f" | Maia={len(test_dataset):,}"
    f" | delta={len(test_dataset) - GPT2_TEST_SEGMENTS:+,}"
)

print()


# ============================================================
# TRAINING BUDGET IMPLIED BY MAIA TOKENIZATION
# ============================================================

ACCUMULATION = 8
EPOCHS = 3

batches_per_epoch = len(
    train_dataset
)

updates_per_epoch = (
    batches_per_epoch
    // ACCUMULATION
)

total_updates = (
    updates_per_epoch
    * EPOCHS
)

remainder = (
    batches_per_epoch
    % ACCUMULATION
)

print("=" * 80)
print("IMPLIED TRAINING BUDGET")
print("=" * 80)
print()

print(
    f"Batches/epoch:          "
    f"{batches_per_epoch:,}"
)

print(
    f"Gradient accumulation:  "
    f"{ACCUMULATION}"
)

print(
    f"Full updates/epoch:      "
    f"{updates_per_epoch:,}"
)

print(
    f"Remainder batches:       "
    f"{remainder}"
)

print(
    f"Total updates (3 epoch): "
    f"{total_updates:,}"
)

print()

print("=" * 80)
print("AUDIT COMPLETE - NO TRAINING PERFORMED")
print("=" * 80)

MAIA LITE STAGE 4 - PRE-TRAINING DATASET AUDIT

Tokenizer:
/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer

Vocabulary:  50257
BOS:         0
PAD:         1
EOS:         2
UNK:         3
MASK:        4

Tokenizer identity: PASS



[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1325 > 1024). Running this sequence through the model will result in indexing errors


Frozen conversations:
  TRAIN: 16,206
  VAL:   901
  TEST:  893

Tokenizing TRAIN with Maia tokenizer...
Tokenizing VALIDATION with Maia tokenizer...
Tokenizing TEST with Maia tokenizer...

MAIA TOKENIZATION RESULTS

TRAIN conversations:     16,206
TRAIN accepted segments: 30,398

VAL conversations:       901
VAL accepted segments:   1,744

TEST conversations:      893
TEST accepted segments:  1,720

COMPARISON WITH GPT-2 MEDIUM STAGE 4

TRAIN: GPT-2=30,011 | Maia=30,398 | delta=+387
VAL:   GPT-2=1,721 | Maia=1,744 | delta=+23
TEST:  GPT-2=1,691 | Maia=1,720 | delta=+29

IMPLIED TRAINING BUDGET

Batches/epoch:          30,398
Gradient accumulation:  8
Full updates/epoch:      3,799
Remainder batches:       6
Total updates (3 epoch): 11,397

AUDIT COMPLETE - NO TRAINING PERFORMED


In [2]:
# ============================================================
# MAIA LITE - SFT A100
# Maia Lite 355M Stage 4 LoRA
# PRE-FLIGHT AUDIT ONLY
#
# This cell:
#   - verifies the frozen Stage 4 corpus
#   - verifies the Maia tokenizer
#   - verifies the Maia base architecture
#   - loads the frozen Maia base
#   - attaches LoRA
#   - builds the response-only datasets
#   - calculates the exact training budget
#
# It DOES NOT:
#   - create an optimizer
#   - perform backward()
#   - update model weights
#   - create training checkpoints
# ============================================================

from pathlib import Path
import hashlib
import json
import math
import sys

import torch

from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
)


# ============================================================
# PATHS
# ============================================================

SFT_PROJECT = Path(
    "/content/drive/MyDrive/Colab_LLMs/maia-sft-validation"
)

MAIA_ROOT = Path(
    "/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3"
)

MAIA_CHECKPOINT = (
    MAIA_ROOT
    / "checkpoints"
    / "checkpoint-216567"
)

MAIA_TOKENIZER = (
    MAIA_ROOT
    / "tokenizer"
)

DATA_DIR = (
    SFT_PROJECT
    / "data"
    / "stage4"
)

TRAIN_FILE = DATA_DIR / "train.jsonl"
VALIDATION_FILE = DATA_DIR / "validation.jsonl"
TEST_FILE = DATA_DIR / "test.jsonl"


# ============================================================
# FROZEN STAGE 4 V4 HASHES
# ============================================================

FROZEN_HASHES = {
    "train.jsonl":
        "ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9",

    "validation.jsonl":
        "5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a",

    "test.jsonl":
        "03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834",
}


# ============================================================
# EXPECTED MAIA IDENTITY
# ============================================================

EXPECTED_TOKENIZER = {
    "vocab": 50257,
    "bos": 0,
    "pad": 1,
    "eos": 2,
    "unk": 3,
    "mask": 4,
}

EXPECTED_MODEL = {
    "vocab_size": 50257,
    "n_positions": 1024,
    "n_embd": 1024,
    "n_layer": 24,
    "n_head": 16,
    "bos_token_id": 0,
    "eos_token_id": 2,
    "pad_token_id": 1,
}

EXPECTED_BASE_PARAMETERS = 354_823_168


# ============================================================
# EXPECTED DATASET RESULTS
# ============================================================

EXPECTED_TRAIN_CONVERSATIONS = 16_206
EXPECTED_VALIDATION_CONVERSATIONS = 901
EXPECTED_TEST_CONVERSATIONS = 893

EXPECTED_TRAIN_SEGMENTS = 30_398
EXPECTED_VALIDATION_SEGMENTS = 1_744
EXPECTED_TEST_SEGMENTS = 1_720


# ============================================================
# TRAINING CONFIGURATION
# ============================================================

MAX_LENGTH = 1024

GRADIENT_ACCUMULATION_STEPS = 8
EPOCHS = 3

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

LORA_TARGET_MODULES = [
    "c_attn",
    "c_proj",
    "c_fc",
]


# ============================================================
# IMPORT PROJECT DATASET
# ============================================================

if str(SFT_PROJECT) not in sys.path:
    sys.path.insert(
        0,
        str(SFT_PROJECT),
    )

from src.dataset_stage4 import ResponseOnlyDataset


# ============================================================
# UTILITIES
# ============================================================

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                1024 * 1024
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def read_jsonl(path):
    rows = []

    with path.open(
        "r",
        encoding="utf-8",
    ) as handle:

        for line in handle:
            line = line.strip()

            if line:
                rows.append(
                    json.loads(line)
                )

    return rows


def pass_fail(condition):
    return "PASS" if condition else "FAIL"


# ============================================================
# HEADER
# ============================================================

print("=" * 80)
print("MAIA LITE 355M - STAGE 4 LoRA A100")
print("PRE-FLIGHT AUDIT ONLY")
print("=" * 80)
print()

print("Project:")
print(" ", SFT_PROJECT)

print()
print("Base:")
print(" ", MAIA_CHECKPOINT)

print()
print("Tokenizer:")
print(" ", MAIA_TOKENIZER)

print()


# ============================================================
# ENVIRONMENT
# ============================================================

print("=" * 80)
print("A100 ENVIRONMENT")
print("=" * 80)
print()

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available."
    )

GPU_NAME = torch.cuda.get_device_name(0)

GPU_MEMORY_GIB = (
    torch.cuda
    .get_device_properties(0)
    .total_memory
    / 1024**3
)

print("GPU:          ", GPU_NAME)
print(
    "GPU memory:   ",
    f"{GPU_MEMORY_GIB:.2f} GiB",
)
print("PyTorch:      ", torch.__version__)
print("CUDA:         ", torch.version.cuda)
print(
    "BF16 support: ",
    torch.cuda.is_bf16_supported(),
)

if "A100" not in GPU_NAME.upper():
    raise RuntimeError(
        f"Expected NVIDIA A100, got: {GPU_NAME}"
    )

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "A100 BF16 support check failed."
    )

print()
print("A100 environment: PASS")
print()


# ============================================================
# REQUIRED FILES
# ============================================================

print("=" * 80)
print("REQUIRED FILES")
print("=" * 80)
print()

required_paths = [
    MAIA_CHECKPOINT,
    MAIA_CHECKPOINT / "config.json",
    MAIA_TOKENIZER,
    TRAIN_FILE,
    VALIDATION_FILE,
    TEST_FILE,
]

for path in required_paths:

    exists = path.exists()

    print(
        f"{pass_fail(exists):4s}  {path}"
    )

    if not exists:
        raise RuntimeError(
            f"Required path missing: {path}"
        )

print()
print("Required files: PASS")
print()


# ============================================================
# FROZEN CORPUS HASHES
# ============================================================

print("=" * 80)
print("FROZEN STAGE 4 V4 CORPUS")
print("=" * 80)
print()

for filename, expected_hash in FROZEN_HASHES.items():

    path = DATA_DIR / filename

    actual_hash = sha256_file(
        path
    )

    print(filename)
    print("  expected:", expected_hash)
    print("  actual:  ", actual_hash)

    if actual_hash != expected_hash:
        raise RuntimeError(
            f"Frozen corpus hash mismatch: {filename}"
        )

    print("  status:   PASS")
    print()

print("Frozen corpus verification: PASS")
print()


# ============================================================
# TOKENIZER
# ============================================================

print("=" * 80)
print("MAIA TOKENIZER")
print("=" * 80)
print()

tokenizer = AutoTokenizer.from_pretrained(
    MAIA_TOKENIZER,
    local_files_only=True,
)

actual_tokenizer = {
    "vocab":
        len(tokenizer),

    "bos":
        tokenizer.bos_token_id,

    "pad":
        tokenizer.pad_token_id,

    "eos":
        tokenizer.eos_token_id,

    "unk":
        tokenizer.unk_token_id,

    "mask":
        tokenizer.mask_token_id,
}

for key in EXPECTED_TOKENIZER:

    expected = EXPECTED_TOKENIZER[key]
    actual = actual_tokenizer[key]

    print(
        f"{key:8s}: "
        f"{actual!s:8s} "
        f"expected={expected} "
        f"{pass_fail(actual == expected)}"
    )

if actual_tokenizer != EXPECTED_TOKENIZER:
    raise RuntimeError(
        "Maia tokenizer identity mismatch."
    )

print()
print("Maia tokenizer identity: PASS")
print()


# ============================================================
# MODEL CONFIGURATION
# ============================================================

print("=" * 80)
print("MAIA BASE ARCHITECTURE")
print("=" * 80)
print()

config = AutoConfig.from_pretrained(
    MAIA_CHECKPOINT,
    local_files_only=True,
)

actual_model = {
    key:
        getattr(
            config,
            key,
            None,
        )

    for key
    in EXPECTED_MODEL
}

for key in EXPECTED_MODEL:

    expected = EXPECTED_MODEL[key]
    actual = actual_model[key]

    print(
        f"{key:16s}: "
        f"{actual!s:8s} "
        f"expected={expected} "
        f"{pass_fail(actual == expected)}"
    )

if actual_model != EXPECTED_MODEL:
    raise RuntimeError(
        "Maia Lite architecture identity mismatch."
    )

print()
print("Maia architecture identity: PASS")
print()


# ============================================================
# LOAD BASE MODEL
# ============================================================

print("=" * 80)
print("LOADING FROZEN MAIA BASE")
print("=" * 80)
print()

base_model = (
    AutoModelForCausalLM
    .from_pretrained(
        MAIA_CHECKPOINT,
        local_files_only=True,
        dtype=torch.float32,
    )
)

base_parameters = sum(
    parameter.numel()
    for parameter
    in base_model.parameters()
)

print(
    "Base parameters:",
    f"{base_parameters:,}",
)

if (
    base_parameters
    != EXPECTED_BASE_PARAMETERS
):
    raise RuntimeError(
        "Unexpected Maia Lite parameter count."
    )

print(
    "Expected:       ",
    f"{EXPECTED_BASE_PARAMETERS:,}",
)

print()
print("Maia base parameter count: PASS")
print()


# ============================================================
# RESPONSE-ONLY DATASETS
# ============================================================

print("=" * 80)
print("RESPONSE-ONLY DATASET")
print("=" * 80)
print()

train_rows = read_jsonl(
    TRAIN_FILE
)

validation_rows = read_jsonl(
    VALIDATION_FILE
)

test_rows = read_jsonl(
    TEST_FILE
)

print("Conversations:")

print(
    f"  TRAIN: {len(train_rows):,}"
)

print(
    f"  VAL:   {len(validation_rows):,}"
)

print(
    f"  TEST:  {len(test_rows):,}"
)

if (
    len(train_rows)
    != EXPECTED_TRAIN_CONVERSATIONS
):
    raise RuntimeError(
        "Unexpected TRAIN conversation count."
    )

if (
    len(validation_rows)
    != EXPECTED_VALIDATION_CONVERSATIONS
):
    raise RuntimeError(
        "Unexpected VALIDATION conversation count."
    )

if (
    len(test_rows)
    != EXPECTED_TEST_CONVERSATIONS
):
    raise RuntimeError(
        "Unexpected TEST conversation count."
    )

print()
print("Tokenizing TRAIN...")

train_dataset = ResponseOnlyDataset(
    train_rows,
    tokenizer,
    MAX_LENGTH,
)

print("Tokenizing VALIDATION...")

validation_dataset = ResponseOnlyDataset(
    validation_rows,
    tokenizer,
    MAX_LENGTH,
)

print("Tokenizing TEST...")

test_dataset = ResponseOnlyDataset(
    test_rows,
    tokenizer,
    MAX_LENGTH,
)

train_segments = len(
    train_dataset
)

validation_segments = len(
    validation_dataset
)

test_segments = len(
    test_dataset
)

print()
print("Accepted response-only segments:")

print(
    f"  TRAIN: {train_segments:,}"
)

print(
    f"  VAL:   {validation_segments:,}"
)

print(
    f"  TEST:  {test_segments:,}"
)

if (
    train_segments
    != EXPECTED_TRAIN_SEGMENTS
):
    raise RuntimeError(
        "Unexpected TRAIN segment count."
    )

if (
    validation_segments
    != EXPECTED_VALIDATION_SEGMENTS
):
    raise RuntimeError(
        "Unexpected VALIDATION segment count."
    )

if (
    test_segments
    != EXPECTED_TEST_SEGMENTS
):
    raise RuntimeError(
        "Unexpected TEST segment count."
    )

print()
print("Response-only dataset identity: PASS")
print()


# ============================================================
# LoRA
# ============================================================

print("=" * 80)
print("LoRA ATTACHMENT")
print("=" * 80)
print()

base_model.config.pad_token_id = (
    tokenizer.pad_token_id
)

base_model.config.use_cache = False

base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,

    # Same as the canonical GPT-2/A100 control.
    fan_in_fan_out=True,
)

model = get_peft_model(
    base_model,
    lora_config,
)

model.config.use_cache = False

total_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)

trainable_percentage = (
    100.0
    * trainable_parameters
    / total_parameters
)

print("LoRA:")
print("  rank:                  ", LORA_R)
print("  alpha:                 ", LORA_ALPHA)
print("  dropout:               ", LORA_DROPOUT)
print(
    "  target modules:        ",
    ", ".join(
        LORA_TARGET_MODULES
    ),
)
print("  fan_in_fan_out:         True")
print("  use_cache:              ", model.config.use_cache)
print(
    "  gradient checkpointing:",
    model.is_gradient_checkpointing,
)

print()
print(
    "Total parameters:       ",
    f"{total_parameters:,}",
)

print(
    "Trainable parameters:   ",
    f"{trainable_parameters:,}",
)

print(
    "Trainable percentage:   ",
    f"{trainable_percentage:.4f}%",
)

if trainable_parameters <= 0:
    raise RuntimeError(
        "LoRA has no trainable parameters."
    )

print()
print("LoRA attachment: PASS")
print()


# ============================================================
# EXACT TRAINING BUDGET
# ============================================================

print("=" * 80)
print("EXACT TRAINING BUDGET")
print("=" * 80)
print()

batches_per_epoch = (
    train_segments
)

updates_per_epoch = math.ceil(
    batches_per_epoch
    / GRADIENT_ACCUMULATION_STEPS
)

total_updates = (
    updates_per_epoch
    * EPOCHS
)

warmup_updates = max(
    1,
    int(
        total_updates
        * WARMUP_RATIO
    ),
)

remainder_batches = (
    batches_per_epoch
    % GRADIENT_ACCUMULATION_STEPS
)

print(
    "Batches/epoch:          ",
    f"{batches_per_epoch:,}",
)

print(
    "Gradient accumulation:  ",
    GRADIENT_ACCUMULATION_STEPS,
)

print(
    "Effective batch:        ",
    GRADIENT_ACCUMULATION_STEPS,
)

print(
    "Full groups of 8:       ",
    f"{batches_per_epoch // GRADIENT_ACCUMULATION_STEPS:,}",
)

print(
    "Final partial group:    ",
    f"{remainder_batches} batches",
)

print(
    "Updates/epoch:          ",
    f"{updates_per_epoch:,}",
)

print(
    "Epochs:                 ",
    EPOCHS,
)

print(
    "Total updates:          ",
    f"{total_updates:,}",
)

print(
    "Warmup ratio:           ",
    WARMUP_RATIO,
)

print(
    "Warmup updates:         ",
    f"{warmup_updates:,}",
)

print(
    "Learning rate:          ",
    LEARNING_RATE,
)

print(
    "Weight decay:           ",
    WEIGHT_DECAY,
)

print(
    "Max gradient norm:      ",
    MAX_GRAD_NORM,
)


# ============================================================
# EXPECTED BUDGET AUDIT
# ============================================================

if batches_per_epoch != 30_398:
    raise RuntimeError(
        "Expected 30,398 batches/epoch."
    )

if updates_per_epoch != 3_800:
    raise RuntimeError(
        "Expected 3,800 updates/epoch."
    )

if total_updates != 11_400:
    raise RuntimeError(
        "Expected 11,400 total updates."
    )

if warmup_updates != 570:
    raise RuntimeError(
        "Expected 570 warmup updates."
    )

if remainder_batches != 6:
    raise RuntimeError(
        "Expected a final partial group of 6 batches."
    )

print()
print("Training budget: PASS")
print()


# ============================================================
# CLEANUP
# ============================================================

del model
del base_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ============================================================
# FINAL RESULT
# ============================================================

print()
print("=" * 80)
print("MAIA LITE LoRA A100 PRE-FLIGHT: PASS")
print("=" * 80)
print()

print("Expected training experiment:")
print("  Base:                   Maia Lite Base 355M")
print("  Checkpoint:             checkpoint-216567")
print("  Precision:              FP32")
print("  Context:                1024")
print("  Training segments:      30,398")
print("  Validation segments:    1,744")
print("  Test segments:          1,720")
print("  Micro batch:            1")
print("  Gradient accumulation:  8")
print("  Updates/epoch:          3,800")
print("  Epochs:                 3")
print("  Total updates:          11,400")
print("  Warmup updates:         570")
print("  LoRA r/alpha/dropout:   16 / 32 / 0.05")
print()
print("NO TRAINING WAS PERFORMED.")

MAIA LITE 355M - STAGE 4 LoRA A100
PRE-FLIGHT AUDIT ONLY

Project:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation

Base:
  /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/checkpoints/checkpoint-216567

Tokenizer:
  /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer

A100 ENVIRONMENT

GPU:           NVIDIA A100-SXM4-80GB
GPU memory:    79.25 GiB
PyTorch:       2.11.0+cu130
CUDA:          13.0
BF16 support:  True

A100 environment: PASS

REQUIRED FILES

PASS  /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/checkpoints/checkpoint-216567
PASS  /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/checkpoints/checkpoint-216567/config.json
PASS  /content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer
PASS  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/data/stage4/train.jsonl
PASS  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/data/stage4/validation.jsonl
PASS  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/data/stage4/test.jsonl

Required files: PASS

FROZEN STA

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Base parameters: 354,823,168
Expected:        354,823,168

Maia base parameter count: PASS

RESPONSE-ONLY DATASET



[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1325 > 1024). Running this sequence through the model will result in indexing errors


Conversations:
  TRAIN: 16,206
  VAL:   901
  TEST:  893

Tokenizing TRAIN...
Tokenizing VALIDATION...
Tokenizing TEST...

Accepted response-only segments:
  TRAIN: 30,398
  VAL:   1,744
  TEST:  1,720

Response-only dataset identity: PASS

LoRA ATTACHMENT

LoRA:
  rank:                   16
  alpha:                  32
  dropout:                0.05
  target modules:         c_attn, c_proj, c_fc
  fan_in_fan_out:         True
  use_cache:               False
  gradient checkpointing: True

Total parameters:        361,114,624
Trainable parameters:    6,291,456
Trainable percentage:    1.7422%

LoRA attachment: PASS

EXACT TRAINING BUDGET

Batches/epoch:           30,398
Gradient accumulation:   8
Effective batch:         8
Full groups of 8:        3,799
Final partial group:     6 batches
Updates/epoch:           3,800
Epochs:                  3
Total updates:           11,400
Warmup ratio:            0.05
Warmup updates:          570
Learning rate:           2e-05
Weight decay:       

In [5]:
# ============================================================
# MAIA LITE - SFT A100
# Remove incompatible optional torchao dependency
# ============================================================

import subprocess
import sys

print("Removing optional incompatible torchao...")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=True,
)

print()
print("torchao removed successfully.")
print()
print("IMPORTANT:")
print("Restart the Colab runtime now.")
print("Then rerun the environment/setup cells and the pre-flight audit.")

Removing optional incompatible torchao...

torchao removed successfully.

IMPORTANT:
Restart the Colab runtime now.
Then rerun the environment/setup cells and the pre-flight audit.


In [ ]:
# ============================================================
# MAIA LITE - SFT A100
# Maia Lite 355M Stage 4 LoRA
# Real-Time Output
# ============================================================

import os
import subprocess
import sys
from pathlib import Path


PROJECT_DIR = Path(
    "/content/drive/MyDrive/Colab_LLMs/maia-sft-validation"
)

SCRIPT_FILE = (
    PROJECT_DIR
    / "src"
    / "train_stage4_maia_lora_a100.py"
)


# ============================================================
# PRE-LAUNCH CHECKS
# ============================================================

print("=" * 80)
print("MAIA LITE 355M - STAGE 4 LoRA A100")
print("TRAINING LAUNCHER")
print("=" * 80)
print()

if not PROJECT_DIR.exists():
    raise RuntimeError(
        f"Project directory not found: {PROJECT_DIR}"
    )

if not SCRIPT_FILE.exists():
    raise RuntimeError(
        f"Training script not found: {SCRIPT_FILE}"
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available."
    )

gpu_name = torch.cuda.get_device_name(0)

if "A100" not in gpu_name.upper():
    raise RuntimeError(
        f"Expected NVIDIA A100, got: {gpu_name}"
    )

print("Project:")
print(" ", PROJECT_DIR)

print()
print("Script:")
print(" ", SCRIPT_FILE)

print()
print("GPU:")
print(" ", gpu_name)

print()
print("Launching Maia Lite LoRA training...")
print()


# ============================================================
# RUN TRAINING WITH UNBUFFERED REAL-TIME OUTPUT
# ============================================================

os.chdir(PROJECT_DIR)

command = [
    sys.executable,
    "-u",
    "-m",
    "src.train_stage4_maia_lora_a100",
]

process = subprocess.Popen(
    command,
    cwd=str(PROJECT_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    assert process.stdout is not None

    for line in process.stdout:
        print(
            line,
            end="",
            flush=True,
        )

finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()


# ============================================================
# RESULT
# ============================================================

print()
print("=" * 80)
print("PROCESS EXIT CODE:", return_code)
print("=" * 80)

if return_code != 0:
    raise RuntimeError(
        f"Maia Lite LoRA training failed "
        f"with exit code {return_code}."
    )

print()
print(
    "MAIA LITE 355M LoRA A100: "
    "COMPLETED SUCCESSFULLY"
)

MAIA LITE 355M - STAGE 4 LoRA A100
TRAINING LAUNCHER

Project:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation

Script:
  /content/drive/MyDrive/Colab_LLMs/maia-sft-validation/src/train_stage4_maia_lora_a100.py

GPU:
  NVIDIA A100-SXM4-80GB

Launching Maia Lite LoRA training...


VERIFYING FROZEN STAGE 4 V4 CORPUS

train.jsonl
  expected: ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9
  actual:   ee80a1efc74807d354761d9728cd498e9f543c84a5e568e4f3821d2fa48565f9

validation.jsonl
  expected: 5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a
  actual:   5d8ea877e14105e06068f2d7de5ae55d265b4069af2c1b0762b162df1be1347a

test.jsonl
  expected: 03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834
  actual:   03b7289a5841f79e1c95188f22615dca6789846aeddecc1b3c90f9713c918834

Frozen corpus verification: PASS

CUDA device: NVIDIA A100-SXM4-80GB
CUDA memory: 79.25 GiB
BF16 support: True

STAGE 4 LoRA TRAINING

Device: cuda
Model:  /content/drive